# Tratamento e Preparação dos Dados — Projeto Olist

Esta etapa tem como objetivo avaliar a qualidade dos dados, identificar inconsistências e realizar os ajustes necessários para preparar as tabelas para as próximas etapas do projeto.

Os arquivos originais serão preservados, e os tratamentos serão realizados utilizando PySpark.

### Configuração e leitura das tabelas

Inicialização da sessão Spark e leitura dos arquivos CSV, armazenando as nove tabelas em um dicionário para facilitar o processamento.


In [ ]:
!pip install -q pyspark

from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Montar Google Drive
drive.mount('/content/drive')

# Inicializar Spark
spark = (
    SparkSession.builder
    .appName("Projeto Olist - Tratamento")
    .getOrCreate()
)

# Caminho dos arquivos
caminho = "/content/drive/MyDrive/Projeto_Olist/dados/"

# Arquivos do projeto
arquivos = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_orders_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation"
]

# Carregar tabelas
tabelas = {}

for nome in arquivos:
    tabelas[nome] = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("encoding", "UTF-8")
        .csv(caminho + nome + ".csv")
    )

print("Tabelas carregadas:", len(tabelas))

for nome, df in tabelas.items():
    print(f"{nome}: {df.count()} registros")

Mounted at /content/drive
Tabelas carregadas: 9
olist_customers_dataset: 99441 registros
olist_geolocation_dataset: 1000163 registros
olist_order_items_dataset: 112650 registros
olist_order_payments_dataset: 103886 registros
olist_order_reviews_dataset: 104162 registros
olist_orders_dataset: 99441 registros
olist_products_dataset: 32951 registros
olist_sellers_dataset: 3095 registros
product_category_name_translation: 71 registros


### Visualização das tabelas

Exibição dos primeiros registros e da estrutura de cada tabela para conhecer as colunas, os valores e os tipos de dados.

In [ ]:
for nome, df in tabelas.items():
    print("\n" + "=" * 70)
    print(f"TABELA: {nome}")
    print("=" * 70)

    print("\nPrimeiros registros:")
    df.show(5, truncate=False)

    print("\nSchema:")
    df.printSchema()


TABELA: olist_customers_dataset

Primeiros registros:
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151                    |sao paulo            |SP            |
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9acbbff3f3c|8775                    |mogi das cruzes      |SP            |
|4f2d8ab171c80ec8364f7c12e35b23ad|3

### Identificação de valores ausentes

Verificação da quantidade e do percentual de valores nulos em cada coluna das tabelas.

In [ ]:
for nome, df in tabelas.items():
    total = df.count()

    resultado = df.select([
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]).first().asDict()

    print(f"\nTabela: {nome}")

    for coluna, quantidade in resultado.items():
        if quantidade and quantidade > 0:
            percentual = quantidade / total * 100
            print(
                f"{coluna}: {quantidade} nulos "
                f"({percentual:.2f}%)"
            )

    if not any(v for v in resultado.values()):
        print("Nenhum valor nulo encontrado.")


Tabela: olist_customers_dataset
Nenhum valor nulo encontrado.

Tabela: olist_geolocation_dataset
Nenhum valor nulo encontrado.

Tabela: olist_order_items_dataset
Nenhum valor nulo encontrado.

Tabela: olist_order_payments_dataset
Nenhum valor nulo encontrado.

Tabela: olist_order_reviews_dataset
review_id: 1 nulos (0.00%)
order_id: 2236 nulos (2.15%)
review_score: 2380 nulos (2.28%)
review_comment_title: 92157 nulos (88.47%)
review_comment_message: 63079 nulos (60.56%)
review_creation_date: 8764 nulos (8.41%)
review_answer_timestamp: 8785 nulos (8.43%)

Tabela: olist_orders_dataset
order_approved_at: 160 nulos (0.16%)
order_delivered_carrier_date: 1783 nulos (1.79%)
order_delivered_customer_date: 2965 nulos (2.98%)

Tabela: olist_products_dataset
product_category_name: 610 nulos (1.85%)
product_name_lenght: 610 nulos (1.85%)
product_description_lenght: 610 nulos (1.85%)
product_photos_qty: 610 nulos (1.85%)
product_weight_g: 2 nulos (0.01%)
product_length_cm: 2 nulos (0.01%)
product_h

### Investigação dos valores ausentes

Análise dos registros incompletos para compreender as causas das ausências e definir o tratamento adequado.

In [ ]:
# Conferir quantidade de avaliações
reviews = tabelas["olist_order_reviews_dataset"]

print("Total de avaliações:", reviews.count())

# Exibir avaliações sem identificador, pedido ou nota
reviews.filter(
    F.col("review_id").isNull() |
    F.col("order_id").isNull() |
    F.col("review_score").isNull()
).show(10, truncate=False)


# Verificar datas ausentes por situação do pedido
orders = tabelas["olist_orders_dataset"]

orders.groupBy("order_status").agg(
    F.count("*").alias("total_pedidos"),
    F.sum(F.col("order_approved_at").isNull().cast("int"))
        .alias("sem_aprovacao"),
    F.sum(F.col("order_delivered_carrier_date").isNull().cast("int"))
        .alias("sem_envio"),
    F.sum(F.col("order_delivered_customer_date").isNull().cast("int"))
        .alias("sem_entrega")
).orderBy("order_status").show(truncate=False)


# Exibir produtos com informações ausentes
products = tabelas["olist_products_dataset"]

products.filter(
    F.col("product_category_name").isNull() |
    F.col("product_weight_g").isNull() |
    F.col("product_length_cm").isNull() |
    F.col("product_height_cm").isNull() |
    F.col("product_width_cm").isNull()
).show(10, truncate=False)

Total de avaliações: 104162
+---------------------------------------------------+--------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id                                          |order_id                  |review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------------------------------------------------+--------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|,2018-02-16 00:00:00,2018-02-20 10:52:22           |NULL                      |NULL        |NULL                |NULL                  |NULL                |NULL                   |
|A entrega foi efetuada muito antes do prazo dado.  |NULL                      |NULL        |NULL                |NULL                  |NULL                |NULL                   |
|O produto já começou a ser usado e até o presente  |NULL

### Verificação de inconsistências na leitura das avaliações

Releitura do arquivo de avaliações para identificar possíveis linhas com problemas de formatação no CSV.

In [ ]:
from pyspark.sql.types import StringType

# Criar schema com coluna para identificar linhas malformadas
schema_reviews = (
    tabelas["olist_order_reviews_dataset"].schema
    .add("_corrupt_record", StringType(), True)
)

# Reler o arquivo registrando possíveis inconsistências
reviews_verificacao = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema_reviews)
    .csv(caminho + "olist_order_reviews_dataset.csv")
)

# Materializar a leitura
reviews_verificacao.cache()

print("Total de registros:", reviews_verificacao.count())

print(
    "Registros com problemas de formatação:",
    reviews_verificacao.filter(
        F.col("_corrupt_record").isNotNull()
    ).count()
)

# Exibir exemplos
reviews_verificacao.filter(
    F.col("_corrupt_record").isNotNull()
).select("_corrupt_record").show(10, truncate=False)

Total de registros: 104162
Registros com problemas de formatação: 8849
+------------------------------------------------------------------------------------------------------------------------------------------------+
|_corrupt_record                                                                                                                                 |
+------------------------------------------------------------------------------------------------------------------------------------------------+
|"4b49719c8a200003f700d3d986ea1a19","9d6f15f95d01e79bd1349cc208361f09",4,,"Mas um pouco ,travando...pelo valor ta Boa.                           |
|",2018-02-16 00:00:00,2018-02-20 10:52:22                                                                                                       |
|"fe3db7c069d694bab50cc43463f91608","2ca73e2ff9e3a186ad1e1ffb9b1d9c10",5,,"A compra foi realizada facilmente.                                    |
|A entrega foi efetuada muito antes do prazo da

### Releitura da tabela de avaliações

Releitura do arquivo CSV com suporte a quebras de linha nos comentários, evitando a divisão incorreta dos registros.

In [ ]:
reviews_corrigidas_leitura = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("multiLine", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema_reviews)
    .csv(caminho + "olist_order_reviews_dataset.csv")
)

reviews_corrigidas_leitura.cache()

print("Total de registros:", reviews_corrigidas_leitura.count())

print(
    "Registros com problemas:",
    reviews_corrigidas_leitura.filter(
        F.col("_corrupt_record").isNotNull()
    ).count()
)

reviews_corrigidas_leitura.show(5, truncate=False)

Total de registros: 99249
Registros com problemas: 116
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+---------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|_corrupt_record|
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+---------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                                                

### Inspeção dos registros inconsistentes

Visualização dos registros que ainda apresentam problemas de formatação após a releitura do arquivo CSV.

In [ ]:
reviews_corrigidas_leitura.filter(
    F.col("_corrupt_record").isNotNull()
).select(
    "_corrupt_record"
).show(20, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_corrupt_record                                                                                                                                                                                                                                                                                                                          |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|"10

### Ajuste na leitura do CSV de avaliações

Configuração dos caracteres de aspas e escape para interpretar corretamente os comentários e preservar os registros.


In [ ]:
reviews_teste = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema_reviews)
    .csv(caminho + "olist_order_reviews_dataset.csv")
)

reviews_teste.cache()

print("Total de registros:", reviews_teste.count())

print(
    "Registros com problemas:",
    reviews_teste.filter(
        F.col("_corrupt_record").isNotNull()
    ).count()
)

reviews_teste.filter(
    F.col("_corrupt_record").isNotNull()
).select("_corrupt_record").show(5, truncate=False)

Total de registros: 99224
Registros com problemas: 0
+---------------+
|_corrupt_record|
+---------------+
+---------------+



Verificação da quantidade de registros, identificadores e valores ausentes na tabela de avaliações após o ajuste da leitura do arquivo CSV.

In [ ]:
print("Total de registros:", reviews_teste.count())

print(
    "IDs de avaliação distintos:",
    reviews_teste.select("review_id").distinct().count()
)

print(
    "Registros sem ID de avaliação:",
    reviews_teste.filter(F.col("review_id").isNull()).count()
)

print(
    "Registros sem ID do pedido:",
    reviews_teste.filter(F.col("order_id").isNull()).count()
)

print(
    "Notas de avaliação nulas:",
    reviews_teste.filter(F.col("review_score").isNull()).count()
)

reviews_teste.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message"
).show(10, truncate=False)

Total de registros: 99224
IDs de avaliação distintos: 98410
Registros sem ID de avaliação: 0
Registros sem ID do pedido: 0
Notas de avaliação nulas: 0
+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                        |
+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|7bc2406110b926393

Identificação de IDs de avaliação repetidos e verificação de possíveis diferenças entre os registros associados a cada ID.

In [ ]:
ids_repetidos = (
    reviews_teste
    .groupBy("review_id")
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("order_id").alias("pedidos_distintos"),
        F.countDistinct("review_score").alias("notas_distintas")
    )
    .filter(F.col("quantidade_registros") > 1)
    .orderBy(F.desc("quantidade_registros"))
)

print("Quantidade de IDs repetidos:", ids_repetidos.count())

ids_repetidos.show(20, truncate=False)

Quantidade de IDs repetidos: 789
+--------------------------------+--------------------+-----------------+---------------+
|review_id                       |quantidade_registros|pedidos_distintos|notas_distintas|
+--------------------------------+--------------------+-----------------+---------------+
|32415bbf6e341d5d517080a796f79b5c|3                   |3                |1              |
|69a1068c3128a14994e3e422e4539e04|3                   |3                |1              |
|39b4603793c1c7f5f36d809b4a218664|3                   |3                |1              |
|2172867fd5b1a55f98fe4608e1547b4b|3                   |3                |1              |
|38821b5c496b678cf91acc34892805ad|3                   |3                |1              |
|c444278834184f72b1484dfe47de7f97|3                   |3                |1              |
|0c76e7a547a531e7bf9f0b99cba071c1|3                   |3                |1              |
|3415c9f764e478409e8e0660ae816dd2|3                   |3           

Comparação entre os registros da leitura anterior e da nova leitura para identificar possíveis diferenças na interpretação do arquivo CSV.

In [ ]:
print("Leitura anterior:", reviews_corrigidas_leitura.count())
print("Nova leitura:", reviews_teste.count())

print(
    "IDs distintos - leitura anterior:",
    reviews_corrigidas_leitura.select("review_id").distinct().count()
)

print(
    "IDs distintos - nova leitura:",
    reviews_teste.select("review_id").distinct().count()
)

print("Registros com problemas na leitura anterior:",
      reviews_corrigidas_leitura.filter(
          F.col("_corrupt_record").isNotNull()
      ).count())

print("Registros com problemas na nova leitura:",
      reviews_teste.filter(
          F.col("_corrupt_record").isNotNull()
      ).count())

Leitura anterior: 99249
Nova leitura: 99224
IDs distintos - leitura anterior: 98435
IDs distintos - nova leitura: 98410
Registros com problemas na leitura anterior: 116
Registros com problemas na nova leitura: 0


Investigação das diferenças entre as leituras da tabela de avaliações para identificar possíveis registros afetados pelas configurações de aspas e quebras de linha.

In [ ]:
reviews_sem_multiline = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("multiLine", False)
    .option("mode", "PERMISSIVE")
    .schema(schema_reviews)
    .csv(caminho + "olist_order_reviews_dataset.csv")
)

print("Total sem multiLine:", reviews_sem_multiline.count())

reviews_sem_multiline.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message"
).show(10, truncate=False)

Total sem multiLine: 104162
+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                        |
+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                                    

Teste de configuração alternativa de escape para verificar se é possível preservar todos os registros da tabela de avaliações sem gerar linhas problemáticas.

In [ ]:
reviews_teste_escape = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '\\')
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema_reviews)
    .csv(caminho + "olist_order_reviews_dataset.csv")
)

reviews_teste_escape.cache()

print("Total de registros:", reviews_teste_escape.count())

print(
    "Registros com problemas:",
    reviews_teste_escape.filter(
        F.col("_corrupt_record").isNotNull()
    ).count()
)

reviews_teste_escape.filter(
    F.col("_corrupt_record").isNotNull()
).select("_corrupt_record").show(5, truncate=False)

Total de registros: 99249
Registros com problemas: 116
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_corrupt_record                                                                                                                                                                                                                                                                                                              |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Inspeção do arquivo CSV original para analisar a estrutura das aspas e quebras de linha nos comentários que causaram problemas durante a leitura com PySpark.

In [ ]:
caminho_reviews = caminho + "olist_order_reviews_dataset.csv"

with open(caminho_reviews, "r", encoding="utf-8") as arquivo:
    for numero, linha in enumerate(arquivo, start=1):
        if '10f9300f8637c13e5691f6354205006d' in linha:
            print("Linha encontrada:", numero)
            print(linha)
            break

Linha encontrada: 1685
"10f9300f8637c13e5691f6354205006d","61af52f7cc6266b9aead2cc464aa48f0",4,,"Respondi que sim para ""recebi o produto correto"" por falta de opção. Eu ainda não vi o produto. Não quis abrir a embalagem, pois é presente. Caso haja alguma desconformidade entrarei em contato.",2018-06-09 00:00:00,2018-06-10 11:39:53



Comparação entre linhas físicas e registros lógicos do arquivo CSV para identificar o impacto de comentários que ocupam mais de uma linha.

In [ ]:
with open(caminho_reviews, "r", encoding="utf-8") as arquivo:
    total_linhas_fisicas = sum(1 for linha in arquivo)

print("Total de linhas físicas no arquivo:", total_linhas_fisicas)
print("Total de registros esperados:", 99249)
print(
    "Diferença entre linhas físicas e registros esperados:",
    total_linhas_fisicas - 99249
)

Total de linhas físicas no arquivo: 104720
Total de registros esperados: 99249
Diferença entre linhas físicas e registros esperados: 5471


Teste de leitura do arquivo CSV com reconhecimento automático das aspas e tratamento de registros multilinha, verificando a preservação da quantidade original de avaliações.

In [ ]:
reviews_teste_final = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("multiLine", True)
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema_reviews)
    .csv(caminho_reviews)
)

reviews_teste_final.cache()

print("Total de registros:", reviews_teste_final.count())

print(
    "Registros com problemas:",
    reviews_teste_final.filter(
        F.col("_corrupt_record").isNotNull()
    ).count()
)

Total de registros: 99224
Registros com problemas: 0


Leitura do arquivo original com o módulo CSV do Python, respeitando aspas escapadas e comentários com múltiplas linhas, para validar a quantidade de avaliações antes de carregá-las no PySpark.

In [ ]:
import csv

with open(caminho_reviews, "r", encoding="utf-8", newline="") as arquivo:
    leitor = csv.reader(arquivo, delimiter=",", quotechar='"', doublequote=True)
    registros_reviews = list(leitor)

print("Total de registros, sem cabeçalho:", len(registros_reviews) - 1)
print("Quantidade de colunas no cabeçalho:", len(registros_reviews[0]))

print("\nCabeçalho:")
print(registros_reviews[0])

print("\nQuantidade de colunas do primeiro registro:")
print(len(registros_reviews[1]))

Total de registros, sem cabeçalho: 99224
Quantidade de colunas no cabeçalho: 7

Cabeçalho:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Quantidade de colunas do primeiro registro:
7


Validação da estrutura dos registros do arquivo CSV, verificando se todas as avaliações possuem exatamente sete campos, conforme definido no cabeçalho.

In [ ]:
from collections import Counter

quantidade_colunas = Counter(
    len(registro) for registro in registros_reviews[1:]
)

print("Distribuição da quantidade de campos por registro:")

for colunas, quantidade in sorted(quantidade_colunas.items()):
    print(f"{colunas} colunas: {quantidade} registros")

print("\nTotal de registros:", len(registros_reviews) - 1)

print(
    "Registros com estrutura diferente de 7 colunas:",
    sum(
        quantidade
        for colunas, quantidade in quantidade_colunas.items()
        if colunas != 7
    )
)

Distribuição da quantidade de campos por registro:
7 colunas: 99224 registros

Total de registros: 99224
Registros com estrutura diferente de 7 colunas: 0


Verificação da quantidade de identificadores únicos e repetidos na leitura do arquivo CSV, para comparar a integridade dos registros com os resultados da análise exploratória.

In [ ]:
ids_reviews = [
    registro[0]
    for registro in registros_reviews[1:]
]

print("Total de registros:", len(ids_reviews))
print("IDs distintos:", len(set(ids_reviews)))
print("Registros com IDs repetidos:", len(ids_reviews) - len(set(ids_reviews)))
print("IDs vazios:", sum(1 for identificador in ids_reviews if not identificador))

Total de registros: 99224
IDs distintos: 98410
Registros com IDs repetidos: 814
IDs vazios: 0


Comparação dos identificadores de avaliação entre as duas leituras do arquivo para localizar registros presentes na leitura anterior e ausentes na leitura atual.

In [ ]:
ids_anteriores = (
    reviews_corrigidas_leitura
    .filter(F.col("review_id").isNotNull())
    .select("review_id")
    .distinct()
)

ids_atuais = spark.createDataFrame(
    [(registro[0],) for registro in registros_reviews[1:]],
    ["review_id"]
).distinct()

ids_ausentes = ids_anteriores.join(
    ids_atuais,
    on="review_id",
    how="left_anti"
)

print("IDs distintos na leitura anterior:", ids_anteriores.count())
print("IDs distintos na leitura atual:", ids_atuais.count())
print("IDs ausentes na leitura atual:", ids_ausentes.count())

ids_ausentes.show(30, truncate=False)

IDs distintos na leitura anterior: 98435
IDs distintos na leitura atual: 98410
IDs ausentes na leitura atual: 31
+-------------------------------------------------------------------------------------------+
|review_id                                                                                  |
+-------------------------------------------------------------------------------------------+
|",2018-04-21 00:00:00,2018-04-23 16:00:24\n"fcf75ca63b730e47b8a32e0764e88c73"              |
|No ""mercado livre"" isso não acontece!!                                                   |
|JÁ ENVIEI MENSAGENS E NADA!!!!                                                             |
|A entrega FOI SUPER RÁPIDA bem antes do prazo estipulado.Sou 1 consumidor muito satisfeito"|
|04.10.16 e só prometeram p/ 25.11.16 e ainda não                                           |
|Solicito providências                                                                      |
| O produto mesmo conectado na energia   

Validação dos identificadores de avaliação, considerando apenas valores com o padrão de 32 caracteres hexadecimais, para eliminar falsos identificadores gerados pela leitura incorreta do CSV.

In [ ]:
# Filtrar somente IDs válidos na leitura anterior
ids_anteriores_validos = (
    reviews_corrigidas_leitura
    .filter(F.col("review_id").rlike("^[0-9a-fA-F]{32}$"))
    .select("review_id")
    .distinct()
)

# Filtrar somente IDs válidos na leitura atual
ids_atuais_validos = (
    ids_atuais
    .filter(F.col("review_id").rlike("^[0-9a-fA-F]{32}$"))
    .select("review_id")
    .distinct()
)

print("IDs válidos na leitura anterior:", ids_anteriores_validos.count())
print("IDs válidos na leitura atual:", ids_atuais_validos.count())

# Identificar IDs presentes na leitura anterior e ausentes na atual
ids_ainda_ausentes = ids_anteriores_validos.join(
    ids_atuais_validos,
    on="review_id",
    how="left_anti"
)

print("IDs válidos ausentes na leitura atual:", ids_ainda_ausentes.count())

ids_ainda_ausentes.show(30, truncate=False)

IDs válidos na leitura anterior: 98404
IDs válidos na leitura atual: 98410
IDs válidos ausentes na leitura atual: 0
+---------+
|review_id|
+---------+
+---------+



### Verificação da estrutura da tabela de avaliações

Conferir os tipos de dados e a quantidade de valores nulos da tabela de avaliações após corrigir a leitura do arquivo CSV.

In [ ]:
# Exibir os tipos de dados das colunas
reviews_teste.printSchema()

# Conferir a quantidade de valores nulos por coluna
reviews_teste.select([
    F.count(
        F.when(
            F.col(coluna).isNull(),
            coluna
        )
    ).alias(coluna)
    for coluna in reviews_teste.columns
    if coluna != "_corrupt_record"
]).show()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- _corrupt_record: string (nullable = true)

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        0|       0|           0|               87656|                 58247|                   0|                      0|
+---------+--------+------------+--------------------+----------------------+--------------------+-------------------

### Padronização dos tipos de dados das avaliações
Converter as notas para valores inteiros e as datas para o formato timestamp, além de remover a coluna auxiliar utilizada na validação da leitura.

In [ ]:
reviews_tratadas = (
    reviews_teste
    .drop("_corrupt_record")
    .withColumn(
        "review_score",
        F.col("review_score").cast("int")
    )
    .withColumn(
        "review_creation_date",
        F.to_timestamp("review_creation_date")
    )
    .withColumn(
        "review_answer_timestamp",
        F.to_timestamp("review_answer_timestamp")
    )
)

# Conferir a estrutura após a padronização
reviews_tratadas.printSchema()

# Conferir os primeiros registros
reviews_tratadas.show(5, truncate=False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------

### Validação da tabela de avaliações tratada

Verificar se a padronização manteve a quantidade de registros e se alguma conversão gerou valores nulos inesperados.

In [ ]:
# Conferir a quantidade total de registros
print("Total de registros:", reviews_tratadas.count())

# Conferir valores nulos nas colunas convertidas
reviews_tratadas.select([
    F.count(
        F.when(F.col(coluna).isNull(), coluna)
    ).alias(coluna)
    for coluna in [
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ]
]).show()

# Conferir notas fora do intervalo esperado de 1 a 5
reviews_tratadas.filter(
    ~F.col("review_score").between(1, 5)
).select("review_score").distinct().show()

Total de registros: 99224
+------------+--------------------+-----------------------+
|review_score|review_creation_date|review_answer_timestamp|
+------------+--------------------+-----------------------+
|           0|                   0|                      0|
+------------+--------------------+-----------------------+

+------------+
|review_score|
+------------+
+------------+



### Atualização da tabela de avaliações
Atualizar o dicionário de tabelas para utilizar a versão tratada nas próximas etapas do projeto.

In [ ]:
# Substituir a tabela antiga pela versão tratada
tabelas["olist_order_reviews_dataset"] = reviews_tratadas

# Confirmar a atualização
print(
    "Total de registros na tabela atualizada:",
    tabelas["olist_order_reviews_dataset"].count()
)

Total de registros na tabela atualizada: 99224


### Inspeção da tabela de pedidos
Verificar os tipos de dados e a quantidade de valores nulos em cada coluna da tabela de pedidos, identificando quais ajustes serão necessários.

In [ ]:
# Exibir a estrutura da tabela de pedidos
tabelas["olist_orders_dataset"].printSchema()

# Contar valores nulos por coluna
tabelas["olist_orders_dataset"].select([
    F.count(
        F.when(F.col(coluna).isNull(), coluna)
    ).alias(coluna)
    for coluna in tabelas["olist_orders_dataset"].columns
]).show(truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|0       |0          |0           |0                       |1

### Verificação e padronização dos status dos pedidos

Identificar os valores distintos da coluna `order_status` e verificar se há inconsistências na sua escrita antes de realizar a padronização.

In [ ]:
# Verificar os valores distintos e suas quantidades
tabelas["olist_orders_dataset"].groupBy(
    "order_status"
).count().orderBy(
    "order_status"
).show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|approved    |2    |
|canceled    |625  |
|created     |5    |
|delivered   |96478|
|invoiced    |314  |
|processing  |301  |
|shipped     |1107 |
|unavailable |609  |
+------------+-----+



### Padronização dos status dos pedidos

Remover possíveis espaços extras e garantir que todos os valores da coluna `order_status` estejam em letras minúsculas.

In [ ]:
# Padronizar os valores da coluna order_status
pedidos_tratados = (
    tabelas["olist_orders_dataset"]
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )
)

# Conferir os valores após a padronização
pedidos_tratados.groupBy(
    "order_status"
).count().orderBy(
    "order_status"
).show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|approved    |2    |
|canceled    |625  |
|created     |5    |
|delivered   |96478|
|invoiced    |314  |
|processing  |301  |
|shipped     |1107 |
|unavailable |609  |
+------------+-----+



### Verificação da consistência das datas dos pedidos

Verificar se existem registros em que as datas de aprovação, envio e entrega estejam em uma sequência cronológica incompatível com o fluxo normal de um pedido.

In [ ]:
# Identificar datas fora da sequência cronológica esperada
inconsistencias_datas = pedidos_tratados.filter(
    (
        F.col("order_approved_at").isNotNull() &
        (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
    ) |
    (
        F.col("order_delivered_carrier_date").isNotNull() &
        F.col("order_approved_at").isNotNull() &
        (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
    ) |
    (
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_delivered_carrier_date").isNotNull() &
        (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
    )
)

print(
    "Pedidos com datas fora da sequência cronológica:",
    inconsistencias_datas.count()
)

inconsistencias_datas.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
).show(10, truncate=False)

Pedidos com datas fora da sequência cronológica: 1382
+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|order_id                        |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|
+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+
|dcb36b511fcac050b97cd5c05de84dc3|delivered   |2018-06-07 19:03:12     |2018-06-12 23:31:02|2018-06-11 14:54:00         |2018-06-21 15:34:32          |
|688052146432ef8253587b930b01a06d|delivered   |2018-04-22 08:48:13     |2018-04-24 18:25:22|2018-04-23 19:19:14         |2018-04-24 19:31:58          |
|58d4c4747ee059eeeb865b349b41f53a|delivered   |2018-07-21 12:49:32     |2018-07-26 23:31:53|2018-07-24 12:57:00         |2018-07-25 23:58:19          |
|412fccb2b44a99b36714bca3fef8ad7b|

### Classificação das inconsistências nas datas dos pedidos

Nesta etapa, as inconsistências serão separadas conforme a relação entre as datas de compra, aprovação, envio e entrega. Essa classificação permite identificar quais etapas apresentam divergências cronológicas.

In [ ]:
# Contar inconsistências em cada etapa
inconsistencias_por_etapa = pedidos_tratados.agg(
    F.sum(
        F.when(
            F.col("order_approved_at") <
            F.col("order_purchase_timestamp"),
            1
        ).otherwise(0)
    ).alias("aprovacao_antes_da_compra"),

    F.sum(
        F.when(
            F.col("order_delivered_carrier_date") <
            F.col("order_approved_at"),
            1
        ).otherwise(0)
    ).alias("envio_antes_da_aprovacao"),

    F.sum(
        F.when(
            F.col("order_delivered_customer_date") <
            F.col("order_delivered_carrier_date"),
            1
        ).otherwise(0)
    ).alias("entrega_antes_do_envio")
)

inconsistencias_por_etapa.show(truncate=False)

+-------------------------+------------------------+----------------------+
|aprovacao_antes_da_compra|envio_antes_da_aprovacao|entrega_antes_do_envio|
+-------------------------+------------------------+----------------------+
|0                        |1359                    |23                    |
+-------------------------+------------------------+----------------------+



### Identificação de pedidos com inconsistências nas datas

Será criada uma coluna indicadora para identificar pedidos que apresentam alguma divergência na sequência cronológica das datas, preservando os valores originais.

In [ ]:
# Criar indicador de inconsistência nas datas
pedidos_tratados = pedidos_tratados.withColumn(
    "inconsistencia_datas",
    F.when(
        (
            F.col("order_approved_at").isNotNull() &
            (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
        ) |
        (
            F.col("order_delivered_carrier_date").isNotNull() &
            F.col("order_approved_at").isNotNull() &
            (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
        ) |
        (
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_delivered_carrier_date").isNotNull() &
            (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
        ),
        1
    ).otherwise(0)
)

# Conferir a quantidade de pedidos com e sem inconsistências
pedidos_tratados.groupBy(
    "inconsistencia_datas"
).count().orderBy(
    "inconsistencia_datas"
).show()

+--------------------+-----+
|inconsistencia_datas|count|
+--------------------+-----+
|                   0|98059|
|                   1| 1382|
+--------------------+-----+



### Atualização da tabela de pedidos tratada

Após a padronização dos status e a identificação das inconsistências cronológicas, a tabela tratada será atualizada no dicionário para utilização nas próximas etapas do projeto.

In [ ]:
# Atualizar a tabela de pedidos no dicionário
tabelas["olist_orders_dataset"] = pedidos_tratados

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["olist_orders_dataset"].count()
)

print(
    "Total de colunas:",
    len(tabelas["olist_orders_dataset"].columns)
)

Total de registros: 99441
Total de colunas: 9


### Inspeção da tabela de produtos

Nesta etapa, serão verificados os tipos de dados e a quantidade de valores nulos em cada coluna da tabela de produtos, identificando os campos que precisam de tratamento.

In [ ]:
# Exibir a estrutura da tabela de produtos
tabelas["olist_products_dataset"].printSchema()

# Contar valores nulos por coluna
tabelas["olist_products_dataset"].select([
    F.count(
        F.when(F.col(coluna).isNull(), coluna)
    ).alias(coluna)
    for coluna in tabelas["olist_products_dataset"].columns
]).show(truncate=False)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|0         |610           

### Investigação dos valores ausentes na tabela de produtos

Nesta etapa, serão analisados os registros com informações incompletas para identificar padrões de ausência e definir o tratamento mais adequado, preservando os produtos cadastrados.

In [ ]:
produtos = tabelas["olist_products_dataset"]

# Contar registros com os quatro campos descritivos ausentes
print(
    "Registros sem categoria, nome, descrição e fotos:",
    produtos.filter(
        F.col("product_category_name").isNull() &
        F.col("product_name_lenght").isNull() &
        F.col("product_description_lenght").isNull() &
        F.col("product_photos_qty").isNull()
    ).count()
)

# Exibir os dois registros sem informações físicas
produtos.filter(
    F.col("product_weight_g").isNull() |
    F.col("product_length_cm").isNull() |
    F.col("product_height_cm").isNull() |
    F.col("product_width_cm").isNull()
).show(truncate=False)

Registros sem categoria, nome, descrição e fotos: 610
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|09ff539a621711667c43eba6a3bd8466|bebes                |60                 |865                       |3                 |NULL            |NULL             |NULL             |NULL            |
|5eb564652db742ff8f28759cd8d2652a|NULL                 |NULL               |NULL                      |NULL              |NULL            |NULL             |NULL             

### Verificação de valores inválidos nos produtos

Além dos valores ausentes, serão identificados registros com características numéricas iguais ou inferiores a zero, que podem representar inconsistências nos dados.

In [ ]:
# Colunas numéricas que devem apresentar valores positivos
colunas_numericas = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

# Contar valores iguais ou inferiores a zero em cada coluna
produtos.select([
    F.count(
        F.when(F.col(coluna) <= 0, coluna)
    ).alias(coluna)
    for coluna in colunas_numericas
]).show(truncate=False)

+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|0                  |0                         |0                 |4               |0                |0                |0               |
+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



### Investigação de pesos inválidos

Nesta etapa, serão identificados os produtos com peso igual ou inferior a zero, permitindo verificar se existem registros que precisam de correção ou preservação como valores ausentes.

In [ ]:
# Exibir produtos com peso igual ou inferior a zero
produtos.filter(
    F.col("product_weight_g") <= 0
).select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).show(truncate=False)

+--------------------------------+---------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+----------------+-----------------+-----------------+----------------+
|81781c0fed9fe1ad6e8c81fca1e1cb08|cama_mesa_banho      |0               |30               |25               |30              |
|8038040ee2a71048d4bdbbdc985b69ab|cama_mesa_banho      |0               |30               |25               |30              |
|36ba42dd187055e1fbe943b2d11430ca|cama_mesa_banho      |0               |30               |25               |30              |
|e673e90efa65a5409ff4196c038bb5af|cama_mesa_banho      |0               |30               |25               |30              |
+--------------------------------+---------------------+----------------+-----------------+-----------------+--

### Tratamento dos valores ausentes e inválidos dos produtos

Os pesos iguais a zero serão convertidos em valores nulos, pois representam informações inválidas. Os demais valores ausentes serão preservados. Também será aplicada a padronização dos nomes das categorias, removendo espaços extras e uniformizando letras minúsculas.

In [ ]:
# Tratar valores inválidos e padronizar a categoria
produtos_tratados = (
    produtos
    .withColumn(
        "product_weight_g",
        F.when(
            F.col("product_weight_g") <= 0,
            None
        ).otherwise(F.col("product_weight_g"))
    )
    .withColumn(
        "product_category_name",
        F.when(
            F.col("product_category_name").isNotNull(),
            F.lower(F.trim(F.col("product_category_name")))
        ).otherwise(None)
    )
)

# Conferir os pesos inválidos restantes
produtos_tratados.filter(
    F.col("product_weight_g") <= 0
).count()

0

### Validação da tabela de produtos tratada

Nesta etapa, serão conferidos o total de registros, a quantidade de identificadores distintos e os valores nulos após o tratamento, garantindo que os produtos tenham sido preservados.

In [ ]:
# Conferir quantidade de registros
print("Total de registros:", produtos_tratados.count())

# Conferir identificadores distintos
print(
    "IDs distintos:",
    produtos_tratados.select("product_id").distinct().count()
)

# Conferir valores nulos por coluna
produtos_tratados.select([
    F.count(
        F.when(F.col(coluna).isNull(), coluna)
    ).alias(coluna)
    for coluna in produtos_tratados.columns
]).show(truncate=False)

Total de registros: 32951
IDs distintos: 32951
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|0         |610                  |610                |610                       |610               |6               |2                |2                |2               |
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



### Atualização da tabela de produtos tratada

Após o tratamento e a validação, a tabela de produtos será atualizada no dicionário, preservando os 32.951 registros e os identificadores originais.

In [ ]:
# Atualizar a tabela de produtos no dicionário
tabelas["olist_products_dataset"] = produtos_tratados

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["olist_products_dataset"].count()
)

print(
    "Total de colunas:",
    len(tabelas["olist_products_dataset"].columns)
)

Total de registros: 32951
Total de colunas: 9


### Inspeção da tabela de geolocalização

Nesta etapa, serão verificados os tipos de dados, a quantidade de valores nulos e a estrutura da tabela de geolocalização para identificar os ajustes necessários.

In [ ]:
# Exibir a estrutura da tabela de geolocalização
tabelas["olist_geolocation_dataset"].printSchema()

# Contar valores nulos por coluna
tabelas["olist_geolocation_dataset"].select([
    F.count(
        F.when(F.col(coluna).isNull(), coluna)
    ).alias(coluna)
    for coluna in tabelas["olist_geolocation_dataset"].columns
]).show(truncate=False)

# Conferir o total de registros
print(
    "Total de registros:",
    tabelas["olist_geolocation_dataset"].count()
)

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|0                          |0              |0              |0               |0                |
+---------------------------+---------------+---------------+----------------+-----------------+

Total de registros: 1000163


### Padronização dos dados de geolocalização

Nesta etapa, o CEP será convertido para texto, preservando seu formato de identificador. Os nomes das cidades e dos estados também serão padronizados, removendo espaços extras e uniformizando letras minúsculas.

In [ ]:
# Padronizar os dados de geolocalização
geolocalizacao_tratada = (
    tabelas["olist_geolocation_dataset"]
    .withColumn(
        "geolocation_zip_code_prefix",
        F.lpad(
            F.col("geolocation_zip_code_prefix").cast("string"),
            5,
            "0"
        )
    )
    .withColumn(
        "geolocation_city",
        F.lower(F.trim(F.col("geolocation_city")))
    )
    .withColumn(
        "geolocation_state",
        F.upper(F.trim(F.col("geolocation_state")))
    )
)

# Conferir a nova estrutura e alguns registros
geolocalizacao_tratada.printSchema()
geolocalizacao_tratada.show(5, truncate=False)

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng   |geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|01037                      |-23.54562128115268 |-46.63929204800168|sao paulo       |SP               |
|01046                      |-23.546081127035535|-46.64482029837157|sao paulo       |SP               |
|01046                      |-23.54612896641469 |-46.64295148361138|sao paulo       |SP               |
|01041                      |-23.5443921648681  |-46.63949930627844|sao paulo       |SP               |
|01035          

### Análise de duplicidades na tabela de geolocalização

Nesta etapa, serão identificadas as linhas completamente duplicadas e a quantidade de CEPs distintos. O objetivo é eliminar apenas repetições exatas, preservando diferentes coordenadas associadas ao mesmo CEP.

In [ ]:
# Contar registros totais
total_registros = geolocalizacao_tratada.count()

# Contar registros distintos considerando todas as colunas
total_distintos = geolocalizacao_tratada.distinct().count()

# Contar CEPs distintos
ceps_distintos = (
    geolocalizacao_tratada
    .select("geolocation_zip_code_prefix")
    .distinct()
    .count()
)

print("Total de registros:", total_registros)
print("Registros completamente distintos:", total_distintos)
print("Linhas completamente duplicadas:", total_registros - total_distintos)
print("CEPs distintos:", ceps_distintos)

Total de registros: 1000163
Registros completamente distintos: 738332
Linhas completamente duplicadas: 261831
CEPs distintos: 19015


### Remoção de duplicidades na tabela de geolocalização

Serão removidas somente as linhas completamente duplicadas, considerando todas as colunas. Registros com o mesmo CEP, mas com coordenadas diferentes, serão preservados.

In [ ]:
# Remover duplicidades completas
geolocalizacao_tratada = geolocalizacao_tratada.dropDuplicates()

# Conferir o resultado
print(
    "Total de registros após remoção:",
    geolocalizacao_tratada.count()
)

print(
    "CEPs distintos:",
    geolocalizacao_tratada.select(
        "geolocation_zip_code_prefix"
    ).distinct().count()
)

Total de registros após remoção: 738332
CEPs distintos: 19015


### Validação das coordenadas geográficas

Nesta etapa, serão identificadas coordenadas fora dos limites geográficos válidos, garantindo a consistência dos dados de latitude e longitude.

In [ ]:
# Contar coordenadas inválidas
coordenadas_invalidas = geolocalizacao_tratada.filter(
    (~F.col("geolocation_lat").between(-90, 90)) |
    (~F.col("geolocation_lng").between(-180, 180))
)

print(
    "Registros com coordenadas inválidas:",
    coordenadas_invalidas.count()
)

# Exibir exemplos, caso existam
coordenadas_invalidas.show(10, truncate=False)

Registros com coordenadas inválidas: 0
+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
+---------------------------+---------------+---------------+----------------+-----------------+



### Atualização da tabela de geolocalização tratada

Após a padronização dos campos, remoção das duplicidades completas e validação das coordenadas, a tabela tratada será atualizada no dicionário de dados.

In [ ]:
# Atualizar a tabela no dicionário
tabelas["olist_geolocation_dataset"] = geolocalizacao_tratada

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["olist_geolocation_dataset"].count()
)

print(
    "Total de colunas:",
    len(tabelas["olist_geolocation_dataset"].columns)
)

Total de registros: 738332
Total de colunas: 5


### Inspeção da tabela de clientes

Nesta etapa, serão verificados os tipos de dados e a quantidade de valores nulos em cada coluna da tabela de clientes, identificando possíveis necessidades de tratamento.

In [ ]:
# Exibir a estrutura da tabela de clientes
tabelas["olist_customers_dataset"].printSchema()

# Contar valores nulos por coluna
tabelas["olist_customers_dataset"].select([
    F.count(
        F.when(F.col(coluna).isNull(), coluna)
    ).alias(coluna)
    for coluna in tabelas["olist_customers_dataset"].columns
]).show(truncate=False)

# Conferir o total de registros
print(
    "Total de registros:",
    tabelas["olist_customers_dataset"].count()
)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|0          |0                 |0                       |0            |0             |
+-----------+------------------+------------------------+-------------+--------------+

Total de registros: 99441


### Padronização dos dados dos clientes

Nesta etapa, o CEP será convertido para texto, preservando os zeros à esquerda. Os nomes das cidades serão padronizados em letras minúsculas e as siglas dos estados em letras maiúsculas.

In [ ]:
# Padronizar os dados dos clientes
clientes_tratados = (
    tabelas["olist_customers_dataset"]
    .withColumn(
        "customer_zip_code_prefix",
        F.lpad(
            F.col("customer_zip_code_prefix").cast("string"),
            5,
            "0"
        )
    )
    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "customer_state",
        F.upper(F.trim(F.col("customer_state")))
    )
)

# Conferir a estrutura e os primeiros registros
clientes_tratados.printSchema()
clientes_tratados.show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|09790                   |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|01151                   |sao paulo   

### Verificação de duplicidades na tabela de clientes

Nesta etapa, serão verificadas as duplicidades dos identificadores `customer_id` e `customer_unique_id`, além da quantidade de registros completamente distintos.

In [ ]:
# Conferir quantidade total de registros
print("Total de registros:", clientes_tratados.count())

# Conferir identificadores distintos
print(
    "customer_id distintos:",
    clientes_tratados.select("customer_id").distinct().count()
)

print(
    "customer_unique_id distintos:",
    clientes_tratados.select("customer_unique_id").distinct().count()
)

# Conferir registros completamente distintos
print(
    "Registros completamente distintos:",
    clientes_tratados.distinct().count()
)

Total de registros: 99441
customer_id distintos: 99441
customer_unique_id distintos: 96096
Registros completamente distintos: 99441


### Atualização da tabela de clientes tratada

Após a padronização e a validação dos identificadores, a tabela de clientes será atualizada no dicionário de dados, preservando todos os registros.

In [ ]:
# Atualizar a tabela de clientes no dicionário
tabelas["olist_customers_dataset"] = clientes_tratados

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["olist_customers_dataset"].count()
)

print(
    "Total de colunas:",
    len(tabelas["olist_customers_dataset"].columns)
)

Total de registros: 99441
Total de colunas: 5


### Inspeção da tabela de itens dos pedidos

Nesta etapa, serão analisadas as colunas, os tipos de dados e a quantidade de valores ausentes na tabela de itens dos pedidos, identificando possíveis ajustes necessários.

In [ ]:
# Selecionar a tabela de itens dos pedidos
itens = tabelas["olist_order_items_dataset"]

# Exibir o schema
itens.printSchema()

# Exibir os primeiros registros
itens.show(5, truncate=False)

# Conferir o total de registros
print("Total de registros:", itens.count())

# Verificar valores ausentes por coluna
itens.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in itens.columns
]).show()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee65886

### Verificação de duplicidades e valores numéricos

Nesta etapa, serão verificadas possíveis duplicidades na combinação de `order_id` e `order_item_id`, além de valores zerados ou negativos nos campos de quantidade do item, preço e frete.

In [ ]:
# Verificar quantidade de combinações distintas de pedido e item
print(
    "Combinações distintas de pedido e item:",
    itens.select("order_id", "order_item_id").distinct().count()
)

# Verificar registros duplicados pela chave composta
print(
    "Registros duplicados pela chave composta:",
    itens.groupBy("order_id", "order_item_id")
         .count()
         .filter(F.col("count") > 1)
         .count()
)

# Verificar valores zerados ou negativos
itens.select([
    F.count(
        F.when(F.col(c) <= 0, 1)
    ).alias(c)
    for c in ["order_item_id", "price", "freight_value"]
]).show()

# Verificar valores mínimos e máximos
itens.select(
    F.min("order_item_id").alias("menor_item"),
    F.max("order_item_id").alias("maior_item"),
    F.min("price").alias("menor_preco"),
    F.max("price").alias("maior_preco"),
    F.min("freight_value").alias("menor_frete"),
    F.max("freight_value").alias("maior_frete")
).show()

Combinações distintas de pedido e item: 112650
Registros duplicados pela chave composta: 0
+-------------+-----+-------------+
|order_item_id|price|freight_value|
+-------------+-----+-------------+
|            0|    0|          383|
+-------------+-----+-------------+

+----------+----------+-----------+-----------+-----------+-----------+
|menor_item|maior_item|menor_preco|maior_preco|menor_frete|maior_frete|
+----------+----------+-----------+-----------+-----------+-----------+
|         1|        21|       0.85|     6735.0|        0.0|     409.68|
+----------+----------+-----------+-----------+-----------+-----------+



### Tratamento da tabela de itens dos pedidos

Como não foram encontradas duplicidades ou valores ausentes, a tabela será preservada. Será criada a coluna `valor_total_item`, que representa a soma do preço do produto com o valor do frete. Os registros com frete igual a zero serão mantidos.

In [ ]:
# Criar a tabela de itens tratada
itens_tratados = (
    itens
    .withColumn(
        "valor_total_item",
        F.round(
            F.col("price") + F.col("freight_value"),
            2
        )
    )
)

# Exibir os primeiros registros
itens_tratados.show(5, truncate=False)

# Conferir o schema atualizado
itens_tratados.printSchema()

# Conferir o total de registros e colunas
print("Total de registros:", itens_tratados.count())
print("Total de colunas:", len(itens_tratados.columns))

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+----------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|valor_total_item|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+----------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |72.19           |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.9|19.93        |259.83          |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51032eddd242adc84c38acab88f23d|2018-01-18 14:48:30|199.0|

### Atualização da tabela de itens dos pedidos

Após a validação da nova coluna, a tabela tratada será atualizada no dicionário de dados para utilização nas próximas etapas do projeto.

In [ ]:
# Atualizar a tabela de itens no dicionário
tabelas["olist_order_items_dataset"] = itens_tratados

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["olist_order_items_dataset"].count()
)

print(
    "Total de colunas:",
    len(tabelas["olist_order_items_dataset"].columns)
)

Total de registros: 112650
Total de colunas: 8


### Inspeção da tabela de pagamentos

Nesta etapa, serão analisadas a estrutura, os tipos de dados e a presença de valores ausentes na tabela de pagamentos, identificando possíveis inconsistências que precisam de tratamento.

In [ ]:
# Selecionar a tabela de pagamentos
pagamentos = tabelas["olist_order_payments_dataset"]

# Exibir o schema
pagamentos.printSchema()

# Exibir os primeiros registros
pagamentos.show(5, truncate=False)

# Conferir o total de registros
print("Total de registros:", pagamentos.count())

# Verificar valores ausentes por coluna
pagamentos.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in pagamentos.columns
]).show()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |1                   |24.39        |
|25e8ea4e93396b6fa0d3dd708e76c1bd|1                 |credit_card |1                   |65.71        |
|ba78997921bbcdc1373bb41e913ab953|1                 |credit_card |8                   |107.78       |
|42fdf880ba16b47b59251dd489d4441a|1            

### Verificação de duplicidades e valores inconsistentes nos pagamentos

Nesta etapa, serão verificadas possíveis duplicidades na combinação de `order_id` e `payment_sequential`, a distribuição dos tipos de pagamento e a existência de valores zerados ou negativos nas parcelas e nos pagamentos.

In [ ]:
# Verificar combinações distintas de pedido e sequência de pagamento
print(
    "Combinações distintas:",
    pagamentos.select(
        "order_id",
        "payment_sequential"
    ).distinct().count()
)

# Verificar duplicidades pela chave composta
print(
    "Registros duplicados:",
    pagamentos.groupBy(
        "order_id",
        "payment_sequential"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Verificar os tipos de pagamento
pagamentos.groupBy("payment_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

# Verificar valores zerados ou negativos
pagamentos.select([
    F.count(
        F.when(F.col(c) <= 0, 1)
    ).alias(c)
    for c in [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ]
]).show()

# Verificar valores mínimos e máximos
pagamentos.select(
    F.min("payment_sequential").alias("menor_sequencia"),
    F.max("payment_sequential").alias("maior_sequencia"),
    F.min("payment_installments").alias("menor_parcela"),
    F.max("payment_installments").alias("maior_parcela"),
    F.min("payment_value").alias("menor_pagamento"),
    F.max("payment_value").alias("maior_pagamento")
).show()

Combinações distintas: 103886
Registros duplicados: 0
+------------+-----+
|payment_type|count|
+------------+-----+
|credit_card |76795|
|boleto      |19784|
|voucher     |5775 |
|debit_card  |1529 |
|not_defined |3    |
+------------+-----+

+------------------+--------------------+-------------+
|payment_sequential|payment_installments|payment_value|
+------------------+--------------------+-------------+
|                 0|                   2|            9|
+------------------+--------------------+-------------+

+---------------+---------------+-------------+-------------+---------------+---------------+
|menor_sequencia|maior_sequencia|menor_parcela|maior_parcela|menor_pagamento|maior_pagamento|
+---------------+---------------+-------------+-------------+---------------+---------------+
|              1|             29|            0|           24|            0.0|       13664.08|
+---------------+---------------+-------------+-------------+---------------+---------------+



### Investigação dos registros inconsistentes nos pagamentos

Foram identificados registros com parcelas ou valores de pagamento iguais a zero, além de três ocorrências do tipo `not_defined`. Nesta etapa, esses registros serão examinados individualmente para determinar se representam situações válidas ou inconsistências nos dados.

In [ ]:
# Filtrar registros com possíveis inconsistências
pagamentos_inconsistentes = pagamentos.filter(
    (F.col("payment_installments") <= 0) |
    (F.col("payment_value") <= 0) |
    (F.col("payment_type") == "not_defined")
)

# Exibir os registros encontrados
pagamentos_inconsistentes.orderBy(
    "order_id",
    "payment_sequential"
).show(30, truncate=False)

# Conferir a quantidade de registros
print(
    "Total de registros inconsistentes:",
    pagamentos_inconsistentes.count()
)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|00b1cb0320190ca0daa2c88b35206009|1                 |not_defined |1                   |0.0          |
|1a57108394169c0b47d8f876acc9ba2d|2                 |credit_card |0                   |129.94       |
|45ed6e85398a87c253db47c2d9f48216|3                 |voucher     |1                   |0.0          |
|4637ca194b6387e2d538dc89b124b0ee|1                 |not_defined |1                   |0.0          |
|6ccb433e00daae1283ccc956189c82ae|4                 |voucher     |1                   |0.0          |
|744bade1fcf9ff3f31d860ace076d422|2                 |credit_card |0                   |58.69        |
|8bcbe01d44d147f901cd3192671144db|4                 |voucher     |1               

### Análise dos pagamentos em relação aos pedidos

Os registros inconsistentes serão relacionados às tabelas de pedidos e itens para verificar o status de cada pedido e comparar os valores registrados nos pagamentos com os valores dos produtos e fretes. Essa análise ajudará a determinar quais ocorrências podem ser preservadas e quais precisam ser sinalizadas.

In [ ]:
# Agrupar os pagamentos por pedido
resumo_pagamentos = (
    pagamentos
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_pagamentos"),
        F.count("*").alias("quantidade_pagamentos")
    )
)

# Agrupar os itens por pedido
resumo_itens = (
    itens_tratados
    .groupBy("order_id")
    .agg(
        F.sum("valor_total_item").alias("total_itens")
    )
)

# Relacionar os registros inconsistentes aos pedidos
analise_inconsistencias = (
    pagamentos_inconsistentes
    .join(
        tabelas["olist_orders_dataset"].select(
            "order_id",
            "order_status"
        ),
        on="order_id",
        how="left"
    )
    .join(
        resumo_pagamentos,
        on="order_id",
        how="left"
    )
    .join(
        resumo_itens,
        on="order_id",
        how="left"
    )
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
        "order_status",
        "total_pagamentos",
        "total_itens"
    )
    .orderBy("order_id", "payment_sequential")
)

# Exibir o resultado
analise_inconsistencias.show(30, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+------------+------------------+-----------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|order_status|total_pagamentos  |total_itens|
+--------------------------------+------------------+------------+--------------------+-------------+------------+------------------+-----------+
|00b1cb0320190ca0daa2c88b35206009|1                 |not_defined |1                   |0.0          |canceled    |0.0               |NULL       |
|1a57108394169c0b47d8f876acc9ba2d|2                 |credit_card |0                   |129.94       |delivered   |129.94            |129.94     |
|45ed6e85398a87c253db47c2d9f48216|3                 |voucher     |1                   |0.0          |delivered   |71.14             |71.14      |
|4637ca194b6387e2d538dc89b124b0ee|1                 |not_defined |1                   |0.0          |canceled    |0.0       

### Tratamento da tabela de pagamentos

A tabela não apresenta duplicidades nem valores ausentes. Os registros com pagamentos de valor zero serão preservados, pois foram identificados casos compatíveis com vouchers sem cobrança e pedidos cancelados. Os dois registros de cartão de crédito com quantidade de parcelas igual a zero também serão mantidos, mas receberão uma coluna de sinalização para facilitar futuras análises.

In [ ]:
# Criar a tabela de pagamentos tratada
pagamentos_tratados = (
    pagamentos
    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type")))
    )
    .withColumn(
        "inconsistencia_parcelas",
        F.when(
            F.col("payment_installments") <= 0,
            1
        ).otherwise(0)
    )
)

# Exibir os registros sinalizados
pagamentos_tratados.filter(
    F.col("inconsistencia_parcelas") == 1
).show(truncate=False)

# Conferir a quantidade de registros
print("Total de registros:", pagamentos_tratados.count())

# Conferir a quantidade de inconsistências sinalizadas
print(
    "Registros com parcelas inconsistentes:",
    pagamentos_tratados.filter(
        F.col("inconsistencia_parcelas") == 1
    ).count()
)

# Conferir o schema
pagamentos_tratados.printSchema()

+--------------------------------+------------------+------------+--------------------+-------------+-----------------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|inconsistencia_parcelas|
+--------------------------------+------------------+------------+--------------------+-------------+-----------------------+
|744bade1fcf9ff3f31d860ace076d422|2                 |credit_card |0                   |58.69        |1                      |
|1a57108394169c0b47d8f876acc9ba2d|2                 |credit_card |0                   |129.94       |1                      |
+--------------------------------+------------------+------------+--------------------+-------------+-----------------------+

Total de registros: 103886
Registros com parcelas inconsistentes: 2
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer 

### Atualização da tabela de pagamentos tratada

Após a padronização e a sinalização dos registros com quantidade de parcelas inconsistente, a tabela será atualizada no dicionário de dados, mantendo todos os registros originais.

In [ ]:
# Atualizar a tabela de pagamentos no dicionário
tabelas["olist_order_payments_dataset"] = pagamentos_tratados

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["olist_order_payments_dataset"].count()
)

print(
    "Total de colunas:",
    len(tabelas["olist_order_payments_dataset"].columns)
)

Total de registros: 103886
Total de colunas: 6


### Inspeção da tabela de vendedores

Nesta etapa, serão analisadas a estrutura, os tipos de dados e a presença de valores ausentes na tabela de vendedores, identificando possíveis ajustes de padronização e inconsistências.

In [ ]:
# Selecionar a tabela de vendedores
vendedores = tabelas["olist_sellers_dataset"]

# Exibir o schema
vendedores.printSchema()

# Exibir os primeiros registros
vendedores.show(5, truncate=False)

# Conferir o total de registros
print("Total de registros:", vendedores.count())

# Verificar valores ausentes por coluna
vendedores.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in vendedores.columns
]).show()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------------------+----------------------+-----------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|
+--------------------------------+----------------------+-----------------+------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |rio de janeiro   |RJ          |
|c0f3eea2e14555b6faeea3dd58c1b1c3|4195                  |sao paulo        |SP          |
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |braganca paulista|SP          |
+--------------------------------+----------------------+-----------------+------------+
only showi

### Verificação de duplicidades na tabela de vendedores

Nesta etapa, será verificada a quantidade de identificadores distintos e de registros completamente distintos, a fim de identificar possíveis duplicidades antes da padronização dos dados.

In [ ]:
# Conferir a quantidade total de registros
print("Total de registros:", vendedores.count())

# Conferir identificadores distintos
print(
    "seller_id distintos:",
    vendedores.select("seller_id").distinct().count()
)

# Conferir registros completamente distintos
print(
    "Registros completamente distintos:",
    vendedores.distinct().count()
)

# Verificar valores de CEP fora do intervalo esperado
vendedores.select(
    F.min("seller_zip_code_prefix").alias("menor_cep"),
    F.max("seller_zip_code_prefix").alias("maior_cep")
).show()

Total de registros: 3095
seller_id distintos: 3095
Registros completamente distintos: 3095
+---------+---------+
|menor_cep|maior_cep|
+---------+---------+
|     1001|    99730|
+---------+---------+



### Padronização da tabela de vendedores

Como não foram encontradas duplicidades ou valores ausentes, serão realizados apenas ajustes de padronização. O CEP será convertido para texto com cinco caracteres, preservando zeros à esquerda, enquanto os nomes das cidades serão normalizados e as siglas dos estados serão mantidas em letras maiúsculas.

In [ ]:
# Criar a tabela de vendedores tratada
vendedores_tratados = (
    vendedores
    .withColumn(
        "seller_zip_code_prefix",
        F.lpad(
            F.col("seller_zip_code_prefix").cast("string"),
            5,
            "0"
        )
    )
    .withColumn(
        "seller_city",
        F.lower(F.trim(F.col("seller_city")))
    )
    .withColumn(
        "seller_state",
        F.upper(F.trim(F.col("seller_state")))
    )
)

# Exibir os primeiros registros
vendedores_tratados.show(5, truncate=False)

# Conferir o schema atualizado
vendedores_tratados.printSchema()

# Conferir o total de registros
print("Total de registros:", vendedores_tratados.count())

+--------------------------------+----------------------+-----------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|
+--------------------------------+----------------------+-----------------+------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |rio de janeiro   |RJ          |
|c0f3eea2e14555b6faeea3dd58c1b1c3|04195                 |sao paulo        |SP          |
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |braganca paulista|SP          |
+--------------------------------+----------------------+-----------------+------------+
only showing top 5 rows
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nulla

### Atualização da tabela de vendedores tratada

Após a padronização e a validação dos dados, a tabela de vendedores será atualizada no dicionário de dados, preservando todos os registros.

In [ ]:
# Atualizar a tabela de vendedores no dicionário
tabelas["olist_sellers_dataset"] = vendedores_tratados

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["olist_sellers_dataset"].count()
)

print(
    "Total de colunas:",
    len(tabelas["olist_sellers_dataset"].columns)
)

Total de registros: 3095
Total de colunas: 4


### Inspeção da tabela de tradução das categorias

Nesta etapa, serão analisadas a estrutura, os tipos de dados, os primeiros registros e a presença de valores ausentes na tabela que relaciona os nomes originais das categorias de produtos às suas respectivas traduções.

In [ ]:
# Selecionar a tabela de tradução das categorias
traducao = tabelas["product_category_name_translation"]

# Exibir o schema
traducao.printSchema()

# Exibir os primeiros registros
traducao.show(10, truncate=False)

# Conferir o total de registros
print("Total de registros:", traducao.count())

# Verificar valores ausentes por coluna
traducao.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in traducao.columns
]).show()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

+----------------------+-----------------------------+
|product_category_name |product_category_name_english|
+----------------------+-----------------------------+
|beleza_saude          |health_beauty                |
|informatica_acessorios|computers_accessories        |
|automotivo            |auto                         |
|cama_mesa_banho       |bed_bath_table               |
|moveis_decoracao      |furniture_decor              |
|esporte_lazer         |sports_leisure               |
|perfumaria            |perfumery                    |
|utilidades_domesticas |housewares                   |
|telefonia             |telephony                    |
|relogios_presentes    |watches_gifts                |
+----------------------+-----------------------------+
only showing top 10 rows
Total de registros: 71
+---------------------+-----------------------------+
|product

### Verificação de duplicidades na tabela de tradução

Nesta etapa, serão verificados os nomes originais das categorias, suas respectivas traduções e a quantidade de registros completamente distintos, garantindo que cada categoria tenha uma correspondência adequada.

In [ ]:
# Conferir a quantidade total de registros
print("Total de registros:", traducao.count())

# Verificar categorias originais distintas
print(
    "Categorias originais distintas:",
    traducao.select("product_category_name").distinct().count()
)

# Verificar traduções distintas
print(
    "Traduções distintas:",
    traducao.select("product_category_name_english").distinct().count()
)

# Verificar registros completamente distintos
print(
    "Registros completamente distintos:",
    traducao.distinct().count()
)

# Verificar categorias originais repetidas
print("Categorias originais repetidas:")
traducao.groupBy("product_category_name") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

Total de registros: 71
Categorias originais distintas: 71
Traduções distintas: 71
Registros completamente distintos: 71
Categorias originais repetidas:
+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
+---------------------+-----+



### Padronização da tabela de tradução das categorias

Como não foram identificados valores ausentes ou duplicidades, serão realizados apenas ajustes de padronização textual, removendo espaços desnecessários e convertendo os nomes das categorias e suas traduções para letras minúsculas.

In [ ]:
# Criar a tabela de tradução tratada
traducao_tratada = (
    traducao
    .withColumn(
        "product_category_name",
        F.lower(F.trim(F.col("product_category_name")))
    )
    .withColumn(
        "product_category_name_english",
        F.lower(F.trim(F.col("product_category_name_english")))
    )
)

# Exibir os registros
traducao_tratada.show(10, truncate=False)

# Conferir o schema
traducao_tratada.printSchema()

# Conferir o total de registros
print("Total de registros:", traducao_tratada.count())

+----------------------+-----------------------------+
|product_category_name |product_category_name_english|
+----------------------+-----------------------------+
|beleza_saude          |health_beauty                |
|informatica_acessorios|computers_accessories        |
|automotivo            |auto                         |
|cama_mesa_banho       |bed_bath_table               |
|moveis_decoracao      |furniture_decor              |
|esporte_lazer         |sports_leisure               |
|perfumaria            |perfumery                    |
|utilidades_domesticas |housewares                   |
|telefonia             |telephony                    |
|relogios_presentes    |watches_gifts                |
+----------------------+-----------------------------+
only showing top 10 rows
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

Total de registros: 71


### Atualização da tabela de tradução tratada

Após a padronização e a validação dos dados, a tabela de tradução das categorias será atualizada no dicionário, ficando disponível para utilização nas análises e cruzamentos com a tabela de produtos.

In [ ]:
# Atualizar a tabela de tradução no dicionário
tabelas["product_category_name_translation"] = traducao_tratada

# Confirmar a atualização
print(
    "Total de registros:",
    tabelas["product_category_name_translation"].count()
)

print(
    "Total de colunas:",
    len(tabelas["product_category_name_translation"].columns)
)

Total de registros: 71
Total de colunas: 2


### Validação geral das tabelas tratadas

Nesta etapa, será realizada uma conferência geral das nove tabelas após os tratamentos. Serão verificadas as quantidades de registros e colunas de cada uma, garantindo que as transformações foram aplicadas e que os dados foram preservados.

In [ ]:
# Percorrer todas as tabelas do dicionário
for nome, tabela in tabelas.items():
    print(f"Tabela: {nome}")
    print(f"Registros: {tabela.count()}")
    print(f"Colunas: {len(tabela.columns)}")
    print("-" * 50)

Tabela: olist_customers_dataset
Registros: 99441
Colunas: 5
--------------------------------------------------
Tabela: olist_geolocation_dataset
Registros: 738332
Colunas: 5
--------------------------------------------------
Tabela: olist_order_items_dataset
Registros: 112650
Colunas: 8
--------------------------------------------------
Tabela: olist_order_payments_dataset
Registros: 103886
Colunas: 6
--------------------------------------------------
Tabela: olist_order_reviews_dataset
Registros: 99224
Colunas: 7
--------------------------------------------------
Tabela: olist_orders_dataset
Registros: 99441
Colunas: 9
--------------------------------------------------
Tabela: olist_products_dataset
Registros: 32951
Colunas: 9
--------------------------------------------------
Tabela: olist_sellers_dataset
Registros: 3095
Colunas: 4
--------------------------------------------------
Tabela: product_category_name_translation
Registros: 71
Colunas: 2
------------------------------------

### Validação dos relacionamentos entre as tabelas

Nesta etapa, serão verificadas as correspondências entre os identificadores das tabelas relacionadas. O objetivo é identificar registros que não encontram correspondência nas tabelas de referência, garantindo maior consistência para futuras análises e cruzamentos de dados.

In [ ]:
# Verificar correspondência entre clientes e pedidos
clientes_sem_pedido = (
    tabelas["olist_orders_dataset"]
    .select("customer_id").distinct()
    .join(
        tabelas["olist_customers_dataset"]
        .select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

# Verificar correspondência entre pedidos e itens
itens_sem_pedido = (
    tabelas["olist_order_items_dataset"]
    .select("order_id").distinct()
    .join(
        tabelas["olist_orders_dataset"]
        .select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

# Verificar correspondência entre produtos e itens
produtos_sem_correspondencia = (
    tabelas["olist_order_items_dataset"]
    .select("product_id").distinct()
    .join(
        tabelas["olist_products_dataset"]
        .select("product_id").distinct(),
        on="product_id",
        how="left_anti"
    )
)

# Verificar correspondência entre vendedores e itens
vendedores_sem_correspondencia = (
    tabelas["olist_order_items_dataset"]
    .select("seller_id").distinct()
    .join(
        tabelas["olist_sellers_dataset"]
        .select("seller_id").distinct(),
        on="seller_id",
        how="left_anti"
    )
)

# Verificar correspondência entre pagamentos e pedidos
pagamentos_sem_pedido = (
    tabelas["olist_order_payments_dataset"]
    .select("order_id").distinct()
    .join(
        tabelas["olist_orders_dataset"]
        .select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

# Verificar correspondência entre avaliações e pedidos
avaliacoes_sem_pedido = (
    tabelas["olist_order_reviews_dataset"]
    .select("order_id").distinct()
    .join(
        tabelas["olist_orders_dataset"]
        .select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

# Exibir os resultados
print("Clientes sem correspondência:", clientes_sem_pedido.count())
print("Itens sem pedido correspondente:", itens_sem_pedido.count())
print("Produtos sem correspondência:", produtos_sem_correspondencia.count())
print("Vendedores sem correspondência:", vendedores_sem_correspondencia.count())
print("Pagamentos sem pedido correspondente:", pagamentos_sem_pedido.count())
print("Avaliações sem pedido correspondente:", avaliacoes_sem_pedido.count())

Clientes sem correspondência: 0
Itens sem pedido correspondente: 0
Produtos sem correspondência: 0
Vendedores sem correspondência: 0
Pagamentos sem pedido correspondente: 0
Avaliações sem pedido correspondente: 0


### Auditoria final de valores nulos

Nesta etapa, é realizada uma verificação dos valores nulos em todas as tabelas tratadas. O objetivo é identificar ausências remanescentes e avaliar se elas representam informações naturalmente não preenchidas ou possíveis problemas de qualidade dos dados.

In [ ]:
# Verificar a quantidade de valores nulos em cada coluna de cada tabela
for nome, tabela in tabelas.items():
    resultado = tabela.agg(
        *[
            F.sum(
                F.when(F.col(coluna).isNull(), 1).otherwise(0)
            ).alias(coluna)
            for coluna in tabela.columns
        ]
    ).collect()[0]

    print(f"\nTabela: {nome}")

    nulos_encontrados = False

    for coluna in tabela.columns:
        quantidade = resultado[coluna] or 0

        if quantidade > 0:
            print(f"{coluna}: {quantidade}")
            nulos_encontrados = True

    if not nulos_encontrados:
        print("Nenhum valor nulo encontrado.")


Tabela: olist_customers_dataset
Nenhum valor nulo encontrado.

Tabela: olist_geolocation_dataset
Nenhum valor nulo encontrado.

Tabela: olist_order_items_dataset
Nenhum valor nulo encontrado.

Tabela: olist_order_payments_dataset
Nenhum valor nulo encontrado.

Tabela: olist_order_reviews_dataset
review_comment_title: 87656
review_comment_message: 58247

Tabela: olist_orders_dataset
order_approved_at: 160
order_delivered_carrier_date: 1783
order_delivered_customer_date: 2965

Tabela: olist_products_dataset
product_category_name: 610
product_name_lenght: 610
product_description_lenght: 610
product_photos_qty: 610
product_weight_g: 6
product_length_cm: 2
product_height_cm: 2
product_width_cm: 2

Tabela: olist_sellers_dataset
Nenhum valor nulo encontrado.

Tabela: product_category_name_translation
Nenhum valor nulo encontrado.


### Validação das colunas derivadas

Nesta etapa, serão verificadas as colunas criadas durante o tratamento dos dados, conferindo suas distribuições e a consistência dos valores calculados.

In [ ]:
# Distribuição das inconsistências nas datas dos pedidos
print("Inconsistência nas datas dos pedidos:")
tabelas["olist_orders_dataset"] \
    .groupBy("inconsistencia_datas") \
    .count() \
    .orderBy("inconsistencia_datas") \
    .show()

# Verificar valores totais dos itens
print("Resumo do valor total dos itens:")
tabelas["olist_order_items_dataset"] \
    .select(
        F.min("valor_total_item").alias("valor_minimo"),
        F.max("valor_total_item").alias("valor_maximo"),
        F.avg("valor_total_item").alias("media")
    ) \
    .show()

# Distribuição das inconsistências nas parcelas
print("Inconsistência nas parcelas dos pagamentos:")
tabelas["olist_order_payments_dataset"] \
    .groupBy("inconsistencia_parcelas") \
    .count() \
    .orderBy("inconsistencia_parcelas") \
    .show()

Inconsistência nas datas dos pedidos:
+--------------------+-----+
|inconsistencia_datas|count|
+--------------------+-----+
|                   0|98059|
|                   1| 1382|
+--------------------+-----+

Resumo do valor total dos itens:
+------------+------------+------------------+
|valor_minimo|valor_maximo|             media|
+------------+------------+------------------+
|        6.08|     6929.31|140.64405894362412|
+------------+------------+------------------+

Inconsistência nas parcelas dos pagamentos:
+-----------------------+------+
|inconsistencia_parcelas| count|
+-----------------------+------+
|                      0|103884|
|                      1|     2|
+-----------------------+------+



### Salvamento das tabelas tratadas em CSV

Nesta etapa, as nove tabelas tratadas serão exportadas em formato CSV, com cabeçalhos e codificação UTF-8. Os arquivos originais serão preservados em seu diretório de origem.

In [ ]:
import os

# Diretório de saída
diretorio_saida = "/content/drive/MyDrive/Projeto_Olist/dados_tratados_csv"

# Criar diretório, caso não exista
os.makedirs(diretorio_saida, exist_ok=True)

# Salvar cada tabela tratada em CSV
for nome, tabela in tabelas.items():

    destino = os.path.join(diretorio_saida, nome)

    (
        tabela
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .option("encoding", "UTF-8")
        .csv(destino)
    )

    print(f"Tabela salva: {nome}")

print("\nExportação concluída!")

Tabela salva: olist_customers_dataset
Tabela salva: olist_geolocation_dataset
Tabela salva: olist_order_items_dataset
Tabela salva: olist_order_payments_dataset
Tabela salva: olist_order_reviews_dataset
Tabela salva: olist_orders_dataset
Tabela salva: olist_products_dataset
Tabela salva: olist_sellers_dataset
Tabela salva: product_category_name_translation

Exportação concluída!


### Verificação dos arquivos exportados

Nesta etapa, será verificada a existência dos arquivos CSV e comparada a quantidade de registros exportados com a quantidade de registros das tabelas tratadas.

In [ ]:
import os

diretorio_saida = "/content/drive/MyDrive/Projeto_Olist/dados_tratados_csv"

for nome, tabela in tabelas.items():

    pasta = os.path.join(diretorio_saida, nome)

    arquivos_csv = [
        arquivo for arquivo in os.listdir(pasta)
        if arquivo.endswith(".csv")
    ]

    print(f"Tabela: {nome}")
    print(f"Arquivos CSV encontrados: {len(arquivos_csv)}")
    print(f"Registros na tabela original tratada: {tabela.count():,}")
    print("-" * 50)

Tabela: olist_customers_dataset
Arquivos CSV encontrados: 1
Registros na tabela original tratada: 99,441
--------------------------------------------------
Tabela: olist_geolocation_dataset
Arquivos CSV encontrados: 1
Registros na tabela original tratada: 738,332
--------------------------------------------------
Tabela: olist_order_items_dataset
Arquivos CSV encontrados: 1
Registros na tabela original tratada: 112,650
--------------------------------------------------
Tabela: olist_order_payments_dataset
Arquivos CSV encontrados: 1
Registros na tabela original tratada: 103,886
--------------------------------------------------
Tabela: olist_order_reviews_dataset
Arquivos CSV encontrados: 1
Registros na tabela original tratada: 99,224
--------------------------------------------------
Tabela: olist_orders_dataset
Arquivos CSV encontrados: 1
Registros na tabela original tratada: 99,441
--------------------------------------------------
Tabela: olist_products_dataset
Arquivos CSV encontr

### Cálculo dos indicadores gerais do e-commerce

Nesta etapa, serão calculados os principais indicadores de desempenho da operação: quantidade de pedidos, receita total, ticket médio e percentual de entregas realizadas no prazo.

In [ ]:
from pyspark.sql import functions as F

# Carregar as tabelas tratadas
pedidos = tabelas["olist_orders_dataset"]
itens = tabelas["olist_order_items_dataset"]

# 1. Total de pedidos
total_pedidos = pedidos.select(
    F.countDistinct("order_id").alias("total_pedidos")
)

# 2. Receita total
receita_total = itens.agg(
    F.round(F.sum("valor_total_item"), 2).alias("receita_total")
)

# 3. Ticket médio por pedido
ticket_medio = (
    itens.groupBy("order_id")
    .agg(F.sum("valor_total_item").alias("valor_pedido"))
    .agg(F.round(F.avg("valor_pedido"), 2).alias("ticket_medio"))
)

# 4. Percentual de entregas no prazo
entregas = pedidos.filter(
    (F.col("order_status") == "delivered") &
    F.col("order_delivered_customer_date").isNotNull() &
    F.col("order_estimated_delivery_date").isNotNull()
)

resultado_entregas = entregas.agg(
    F.count("*").alias("total_entregues"),
    F.sum(
        F.when(
            F.to_date("order_delivered_customer_date") <=
            F.to_date("order_estimated_delivery_date"),
            1
        ).otherwise(0)
    ).alias("entregues_no_prazo")
)

# Exibir resultados
print("TOTAL DE PEDIDOS")
total_pedidos.show()

print("RECEITA TOTAL")
receita_total.show()

print("TICKET MÉDIO")
ticket_medio.show()

print("ENTREGAS")
resultado_entregas.show()

TOTAL DE PEDIDOS
+-------------+
|total_pedidos|
+-------------+
|        99441|
+-------------+

RECEITA TOTAL
+-------------+
|receita_total|
+-------------+
|1.584355324E7|
+-------------+

TICKET MÉDIO
+------------+
|ticket_medio|
+------------+
|      160.58|
+------------+

ENTREGAS
+---------------+------------------+
|total_entregues|entregues_no_prazo|
+---------------+------------------+
|          96470|             89936|
+---------------+------------------+



### Verificação das datas de entrega

Será verificada a quantidade de pedidos entregues que não possuem alguma das datas necessárias para calcular o cumprimento do prazo.

In [ ]:
pedidos.filter(
    (F.col("order_status") == "delivered") &
    (
        F.col("order_delivered_customer_date").isNull() |
        F.col("order_estimated_delivery_date").isNull()
    )
).select(
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(20, False)

+--------------------------------+-----------------------------+-----------------------------+
|order_id                        |order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+-----------------------------+-----------------------------+
|2d1e2d5bf4dc7227b3bfebb81328c15f|NULL                         |2017-12-18 00:00:00          |
|f5dd62b788049ad9fc0526e3ad11a097|NULL                         |2018-07-16 00:00:00          |
|2ebdfc4f15f23b91474edf87475f108e|NULL                         |2018-07-30 00:00:00          |
|e69f75a717d64fc5ecdfae42b2e8e086|NULL                         |2018-07-30 00:00:00          |
|0d3268bad9b086af767785e3f0fc0133|NULL                         |2018-07-24 00:00:00          |
|2d858f451373b04fb5c984a1cc2defaf|NULL                         |2017-06-23 00:00:00          |
|ab7c89dc1bf4a1ead9d6ec1ec8968a84|NULL                         |2018-06-26 00:00:00          |
|20edc82cf5400ce95e1afacc25798b31|NULL            

### Consolidação dos indicadores gerais

Nesta etapa, os indicadores calculados serão reunidos em uma única tabela, que servirá como base para os cartões de desempenho do dashboard.

In [ ]:
# Consolidar os quatro indicadores em uma tabela
indicadores_gerais = (
    total_pedidos
    .crossJoin(receita_total)
    .crossJoin(ticket_medio)
    .crossJoin(
        resultado_entregas.withColumn(
            "percentual_entregas_no_prazo",
            F.round(
                F.col("entregues_no_prazo") /
                F.col("total_entregues") * 100,
                2
            )
        )
    )
)

indicadores_gerais.show(truncate=False)

+-------------+-------------+------------+---------------+------------------+----------------------------+
|total_pedidos|receita_total|ticket_medio|total_entregues|entregues_no_prazo|percentual_entregas_no_prazo|
+-------------+-------------+------------+---------------+------------------+----------------------------+
|99441        |1.584355324E7|160.58      |96470          |89936             |93.23                       |
+-------------+-------------+------------+---------------+------------------+----------------------------+



### Exportação dos indicadores gerais

A tabela consolidada de indicadores será exportada em CSV, mantendo os nomes das colunas e a codificação UTF-8 para utilização no Looker Studio.

In [ ]:
# Diretório dos indicadores
diretorio_indicadores = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores"
)

import os
os.makedirs(diretorio_indicadores, exist_ok=True)

# Exportar a tabela
(
    indicadores_gerais
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(diretorio_indicadores + "/indicadores_gerais")
)

print("Indicadores exportados com sucesso!")

Indicadores exportados com sucesso!


### Evolução mensal de pedidos e receita

Nesta etapa, os pedidos serão agrupados por mês de compra e relacionados aos respectivos valores dos itens e fretes. O objetivo é identificar a evolução do volume de pedidos e da receita ao longo do período analisado.

In [ ]:
# Conferir o período disponível na base de pedidos
pedidos.select(
    F.min("order_purchase_timestamp").alias("primeira_compra"),
    F.max("order_purchase_timestamp").alias("ultima_compra")
).show(truncate=False)

+-------------------+-------------------+
|primeira_compra    |ultima_compra      |
+-------------------+-------------------+
|2016-09-04 21:15:19|2018-10-17 17:30:18|
+-------------------+-------------------+



### Agrupamento mensal dos pedidos e da receita

Os dados serão agrupados pelo mês de realização da compra. A receita será calculada a partir dos itens associados a cada pedido, evitando duplicidades no relacionamento entre as tabelas.

In [ ]:
# Agrupar os valores dos itens por pedido
receita_por_pedido = (
    itens.groupBy("order_id")
    .agg(
        F.sum("valor_total_item").alias("receita_pedido")
    )
)

# Associar os pedidos aos seus respectivos valores
pedidos_com_receita = pedidos.join(
    receita_por_pedido,
    on="order_id",
    how="left"
)

# Criar a tabela de evolução mensal
evolucao_mensal = (
    pedidos_com_receita
    .withColumn(
        "ano_mes",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .groupBy("ano_mes")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(
            F.sum("receita_pedido"), 2
        ).alias("receita_total")
    )
    .orderBy("ano_mes")
)

# Exibir os resultados
evolucao_mensal.show(30, truncate=False)

+-------+-------------+-------------+
|ano_mes|total_pedidos|receita_total|
+-------+-------------+-------------+
|2016-09|4            |354.75       |
|2016-10|324          |56808.84     |
|2016-12|1            |19.62        |
|2017-01|800          |137188.49    |
|2017-02|1780         |286280.62    |
|2017-03|2682         |432048.59    |
|2017-04|2404         |412422.24    |
|2017-05|3700         |586190.95    |
|2017-06|3245         |502963.04    |
|2017-07|4026         |584971.62    |
|2017-08|4331         |668204.6     |
|2017-09|4285         |720398.91    |
|2017-10|4631         |769312.37    |
|2017-11|7544         |1179143.77   |
|2017-12|5673         |863547.23    |
|2018-01|7269         |1107301.89   |
|2018-02|6728         |986908.96    |
|2018-03|7211         |1155126.82   |
|2018-04|6939         |1159698.04   |
|2018-05|6873         |1149781.82   |
|2018-06|6167         |1022677.11   |
|2018-07|6292         |1058728.03   |
|2018-08|6512         |1003308.47   |
|2018-09|16 

### Investigação dos registros dos meses finais

Será verificada a situação dos pedidos de setembro e outubro de 2018, identificando seus status e possíveis ausências de itens associados.

In [ ]:
# Verificar os pedidos de setembro e outubro de 2018
pedidos_finais = pedidos_com_receita.filter(
    F.col("order_purchase_timestamp") >= F.to_timestamp(
        F.lit("2018-09-01")
    )
)

print("Pedidos por mês e status:")

pedidos_finais.withColumn(
    "ano_mes",
    F.date_format("order_purchase_timestamp", "yyyy-MM")
).groupBy(
    "ano_mes", "order_status"
).count().orderBy(
    "ano_mes", "order_status"
).show(truncate=False)

print("Pedidos sem itens associados:")

pedidos_finais.filter(
    F.col("receita_pedido").isNull()
).select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "receita_pedido"
).show(20, False)

Pedidos por mês e status:
+-------+------------+-----+
|ano_mes|order_status|count|
+-------+------------+-----+
|2018-09|canceled    |15   |
|2018-09|shipped     |1    |
|2018-10|canceled    |4    |
+-------+------------+-----+

Pedidos sem itens associados:
+--------------------------------+------------+------------------------+--------------+
|order_id                        |order_status|order_purchase_timestamp|receita_pedido|
+--------------------------------+------------+------------------------+--------------+
|ed3efbd3a87bea76c2812c66a0b32219|canceled    |2018-09-20 13:54:16     |NULL          |
|ea844c92cf978ea23321fa7fe5871761|canceled    |2018-09-13 09:56:12     |NULL          |
|87ae60ef8b08ae0e5f903cacb53a6904|canceled    |2018-09-11 16:45:54     |NULL          |
|bd35b677fd239386e9861d11ae98ab56|canceled    |2018-09-17 17:21:16     |NULL          |
|4d59cc2a2e7bb6c0a851725f5888a9b5|canceled    |2018-09-06 14:50:07     |NULL          |
|0d3adebce4bebc1f80a7f36e9833f497|ca

### Ajuste e validação da evolução mensal

Os valores de receita ausentes serão convertidos em zero quando não houver itens associados aos pedidos. Para a visualização da tendência mensal, serão utilizados os meses até agosto de 2018, evitando distorções causadas pelos registros dos meses finais da base.

In [ ]:
# Corrigir valores de receita ausentes
evolucao_mensal = evolucao_mensal.withColumn(
    "receita_total",
    F.coalesce(F.col("receita_total"), F.lit(0.0))
)

# Criar a base para o gráfico de evolução mensal
evolucao_mensal_dashboard = (
    evolucao_mensal
    .filter(F.col("ano_mes") <= "2018-08")
    .orderBy("ano_mes")
)

# Conferir os resultados
evolucao_mensal_dashboard.show(30, truncate=False)

print(
    "Quantidade de meses para o dashboard:",
    evolucao_mensal_dashboard.count()
)

+-------+-------------+-------------+
|ano_mes|total_pedidos|receita_total|
+-------+-------------+-------------+
|2016-09|4            |354.75       |
|2016-10|324          |56808.84     |
|2016-12|1            |19.62        |
|2017-01|800          |137188.49    |
|2017-02|1780         |286280.62    |
|2017-03|2682         |432048.59    |
|2017-04|2404         |412422.24    |
|2017-05|3700         |586190.95    |
|2017-06|3245         |502963.04    |
|2017-07|4026         |584971.62    |
|2017-08|4331         |668204.6     |
|2017-09|4285         |720398.91    |
|2017-10|4631         |769312.37    |
|2017-11|7544         |1179143.77   |
|2017-12|5673         |863547.23    |
|2018-01|7269         |1107301.89   |
|2018-02|6728         |986908.96    |
|2018-03|7211         |1155126.82   |
|2018-04|6939         |1159698.04   |
|2018-05|6873         |1149781.82   |
|2018-06|6167         |1022677.11   |
|2018-07|6292         |1058728.03   |
|2018-08|6512         |1003308.47   |
+-------+---

### Exportação da evolução mensal

A tabela mensal será exportada em CSV para utilização no Looker Studio. O arquivo manterá a quantidade de pedidos e o valor total comercializado por mês.

In [ ]:
caminho_evolucao = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/"
    "evolucao_mensal_dashboard"
)

(
    evolucao_mensal_dashboard
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_evolucao)
)

print("Exportação concluída!")
print("Destino:", caminho_evolucao)

Exportação concluída!
Destino: /content/drive/MyDrive/Projeto_Olist/indicadores/evolucao_mensal_dashboard


### Estrutura das tabelas de produtos e categorias

Verificação dos nomes das colunas das duas tabelas. Essa verificação permitirá identificar os campos necessários para relacionar os itens vendidos às categorias dos produtos.

In [ ]:
print("COLUNAS DA TABELA DE PRODUTOS")
print(tabelas["olist_products_dataset"].columns)

print("\nCOLUNAS DA TABELA DE TRADUÇÃO")
print(tabelas["product_category_name_translation"].columns)

COLUNAS DA TABELA DE PRODUTOS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

COLUNAS DA TABELA DE TRADUÇÃO
['product_category_name', 'product_category_name_english']


### Validação do relacionamento entre itens e produtos

A tabela de itens será relacionada à tabela de produtos por meio da coluna `product_id`. A verificação identificará se existem itens sem produto correspondente e se a relação está preservando a quantidade original de registros.

In [ ]:
itens = tabelas["olist_order_items_dataset"]
produtos = tabelas["olist_products_dataset"]

itens_com_produtos = itens.join(
    produtos.select("product_id", "product_category_name"),
    on="product_id",
    how="left"
)

print("Total de itens originais:", itens.count())
print("Total após relacionamento:", itens_com_produtos.count())

print(
    "Itens sem produto correspondente:",
    itens_com_produtos
    .filter(F.col("product_category_name").isNull())
    .count()
)

print(
    "Produtos com IDs duplicados:",
    produtos.groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Total de itens originais: 112650
Total após relacionamento: 112650
Itens sem produto correspondente: 1603
Produtos com IDs duplicados: 0


### Análise de categorias não identificadas

Nesta etapa, serão analisados os 1.603 itens que não possuem uma categoria identificada após o relacionamento com a tabela de produtos. O objetivo é verificar se os produtos existem, mas estão com a categoria não preenchida, ou se há outra situação. Nenhum registro será excluído durante essa análise.

In [ ]:
itens_sem_categoria = itens_com_produtos.filter(
    F.col("product_category_name").isNull()
)

print("Total de itens sem categoria:", itens_sem_categoria.count())

print("\nDistribuição por existência do produto:")
itens_sem_categoria.select(
    F.when(
        F.col("product_id").isNull(),
        "ID do produto ausente"
    ).otherwise("Produto encontrado, categoria nula")
    .alias("situacao")
).groupBy("situacao").count().show()

print("\nExemplos de registros:")
itens_sem_categoria.select(
    "product_id",
    "price",
    "freight_value",
    "valor_total_item"
).show(10, truncate=False)

Total de itens sem categoria: 1603

Distribuição por existência do produto:
+--------------------+-----+
|            situacao|count|
+--------------------+-----+
|Produto encontrad...| 1603|
+--------------------+-----+


Exemplos de registros:
+--------------------------------+------+-------------+----------------+
|product_id                      |price |freight_value|valor_total_item|
+--------------------------------+------+-------------+----------------+
|ff6caf9340512b8bf6d2a2a6df032cfa|7.79  |7.78         |15.57           |
|a9c404971d1a5b1cbc2e4070e02731fd|7.6   |10.96        |18.56           |
|5a848e4ab52fd5445cdc07aab1c40e48|122.99|15.61        |138.6           |
|41eee23c25f7a574dfaf8d5c151dbb12|20.3  |16.79        |37.09           |
|e10758160da97891c2fdcbc35f0f031d|56.0  |14.14        |70.14           |
|76d1a1a9d21ab677a61c3ae34b1b352f|159.77|16.0         |175.77          |
|fbb1cfc2810efabf3235eccf4530f4ae|44.9  |16.11        |61.01           |
|5a848e4ab52fd5445cdc07a

### Verificação da tradução das categorias

A tabela de tradução será relacionada às categorias dos produtos. Será verificado se existem categorias sem tradução correspondente e se há chaves duplicadas que possam causar multiplicação dos registros.

In [ ]:
traducao = tabelas["product_category_name_translation"]

produtos_com_traducao = produtos.join(
    traducao,
    on="product_category_name",
    how="left"
)

print(
    "Total de produtos:",
    produtos.count()
)

print(
    "Total após relacionamento:",
    produtos_com_traducao.count()
)

print(
    "Categorias sem tradução:",
    produtos_com_traducao.filter(
        F.col("product_category_name").isNotNull() &
        F.col("product_category_name_english").isNull()
    ).select("product_category_name").distinct().count()
)

print(
    "Categorias duplicadas na tradução:",
    traducao.groupBy("product_category_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Total de produtos: 32951
Total após relacionamento: 32951
Categorias sem tradução: 2
Categorias duplicadas na tradução: 0


### Identificação das categorias sem tradução

Serão exibidos os nomes das duas categorias que não possuem correspondência na tabela de tradução. Essa verificação permitirá definir como apresentar essas categorias na tabela analítica.

In [ ]:
categorias_sem_traducao = (
    produtos_com_traducao
    .filter(
        F.col("product_category_name").isNotNull() &
        F.col("product_category_name_english").isNull()
    )
    .select("product_category_name")
    .distinct()
    .orderBy("product_category_name")
)

categorias_sem_traducao.show(truncate=False)

+---------------------------------------------+
|product_category_name                        |
+---------------------------------------------+
|pc_gamer                                     |
|portateis_cozinha_e_preparadores_de_alimentos|
+---------------------------------------------+



### Agregação da receita por categoria de produto

Os itens vendidos serão relacionados aos produtos e à tabela de tradução. A receita será agrupada por categoria, utilizando o nome em inglês quando disponível e preservando o nome original nos demais casos. Produtos sem categoria identificada serão classificados como “Categoria não informada”.

In [ ]:
produtos_categorias = (
    produtos
    .join(
        traducao,
        on="product_category_name",
        how="left"
    )
    .withColumn(
        "categoria",
        F.when(
            F.col("product_category_name").isNull(),
            "Categoria não informada"
        ).otherwise(
            F.coalesce(
                F.col("product_category_name_english"),
                F.col("product_category_name")
            )
        )
    )
    .select("product_id", "categoria")
)

itens_com_categoria = itens.join(
    produtos_categorias,
    on="product_id",
    how="left"
)

receita_por_categoria = (
    itens_com_categoria
    .groupBy("categoria")
    .agg(
        F.count("*").alias("total_itens"),
        F.round(F.sum("valor_total_item"), 2).alias("receita_total")
    )
    .orderBy(F.desc("receita_total"))
)

receita_por_categoria.show(40, truncate=False)

+---------------------------------------+-----------+-------------+
|categoria                              |total_itens|receita_total|
+---------------------------------------+-----------+-------------+
|health_beauty                          |9670       |1441248.07   |
|watches_gifts                          |5991       |1305541.61   |
|bed_bath_table                         |11115      |1241681.72   |
|sports_leisure                         |8641       |1156656.48   |
|computers_accessories                  |7827       |1059272.4    |
|furniture_decor                        |8334       |902511.79    |
|housewares                             |6964       |778397.77    |
|cool_stuff                             |3796       |719329.95    |
|auto                                   |4235       |685384.32    |
|garden_tools                           |4347       |584219.21    |
|toys                                   |4117       |561372.55    |
|baby                                   |3065   

### Conferência dos valores da receita por categoria

Será comparada a soma da receita agrupada por categoria com a receita total da tabela de itens. Também será conferida a quantidade total de itens, garantindo que todos os registros estejam representados na agregação.

In [ ]:
total_receita_categorias = receita_por_categoria.agg(
    F.round(F.sum("receita_total"), 2).alias("receita_categorias")
)

total_itens_categorias = receita_por_categoria.agg(
    F.sum("total_itens").alias("total_itens")
)

total_receita_original = itens.agg(
    F.round(F.sum("valor_total_item"), 2).alias("receita_original")
)

total_itens_original = itens.count()

print("Receita agrupada por categoria:")
total_receita_categorias.show()

print("Receita original dos itens:")
total_receita_original.show()

print("Itens agrupados por categoria:")
total_itens_categorias.show()

print("Total de itens original:", total_itens_original)

Receita agrupada por categoria:
+------------------+
|receita_categorias|
+------------------+
|     1.584355324E7|
+------------------+

Receita original dos itens:
+----------------+
|receita_original|
+----------------+
|   1.584355324E7|
+----------------+

Itens agrupados por categoria:
+-----------+
|total_itens|
+-----------+
|     112650|
+-----------+

Total de itens original: 112650


### Exportação da receita por categoria

A tabela de receita por categoria será exportada em CSV para utilização no Looker Studio. O arquivo conterá o nome da categoria, a quantidade de itens vendidos e o valor total comercializado.

In [ ]:
caminho_categoria = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/"
    "receita_por_categoria"
)

(
    receita_por_categoria
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_categoria)
)

print("Exportação concluída!")
print("Destino:", caminho_categoria)

Exportação concluída!
Destino: /content/drive/MyDrive/Projeto_Olist/indicadores/receita_por_categoria


### Distribuição dos pedidos por status

A tabela de pedidos será agrupada pelo status para identificar a quantidade e a participação percentual de cada situação. Essa análise servirá de base para uma visualização da distribuição dos pedidos no dashboard.

In [ ]:
from pyspark.sql.window import Window


pedidos = tabelas["olist_orders_dataset"]

pedidos_por_status = (
    pedidos
    .groupBy("order_status")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos")
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("total_pedidos") /
            F.sum("total_pedidos").over(
                Window.partitionBy()
            ) * 100,
            2
        )
    )
    .orderBy(F.desc("total_pedidos"))
)

pedidos_por_status.show(truncate=False)

+------------+-------------+----------+
|order_status|total_pedidos|percentual|
+------------+-------------+----------+
|delivered   |96478        |97.02     |
|shipped     |1107         |1.11      |
|canceled    |625          |0.63      |
|unavailable |609          |0.61      |
|invoiced    |314          |0.32      |
|processing  |301          |0.3       |
|created     |5            |0.01      |
|approved    |2            |0.0       |
+------------+-------------+----------+



### Exportação da distribuição de pedidos por status

A tabela será exportada em CSV com a quantidade de pedidos e o percentual de participação de cada status.

In [ ]:
caminho_status = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/"
    "pedidos_por_status"
)

(
    pedidos_por_status
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_status)
)

print("Exportação concluída!")
print("Destino:", caminho_status)

Exportação concluída!
Destino: /content/drive/MyDrive/Projeto_Olist/indicadores/pedidos_por_status


### Distribuição geográfica dos pedidos

A tabela de pedidos será relacionada à tabela de clientes por meio da coluna `customer_id`. O estado de cada cliente será utilizado para calcular a quantidade de pedidos por unidade federativa, permitindo identificar as regiões com maior concentração de compras.

In [ ]:
pedidos = tabelas["olist_orders_dataset"]
clientes = tabelas["olist_customers_dataset"]

pedidos_com_estado = pedidos.join(
    clientes.select("customer_id", "customer_state"),
    on="customer_id",
    how="left"
)

pedidos_por_estado = (
    pedidos_com_estado
    .groupBy("customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos")
    )
    .orderBy(F.desc("total_pedidos"))
)

pedidos_por_estado.show(30, truncate=False)

print(
    "Total de pedidos após relacionamento:",
    pedidos_com_estado.select("order_id").distinct().count()
)

print(
    "Pedidos sem estado identificado:",
    pedidos_com_estado.filter(
        F.col("customer_state").isNull()
    ).select("order_id").distinct().count()
)

+--------------+-------------+
|customer_state|total_pedidos|
+--------------+-------------+
|SP            |41746        |
|RJ            |12852        |
|MG            |11635        |
|RS            |5466         |
|PR            |5045         |
|SC            |3637         |
|BA            |3380         |
|DF            |2140         |
|ES            |2033         |
|GO            |2020         |
|PE            |1652         |
|CE            |1336         |
|PA            |975          |
|MT            |907          |
|MA            |747          |
|MS            |715          |
|PB            |536          |
|PI            |495          |
|RN            |485          |
|AL            |413          |
|SE            |350          |
|TO            |280          |
|RO            |253          |
|AM            |148          |
|AC            |81           |
|AP            |68           |
|RR            |46           |
+--------------+-------------+

Total de pedidos após relacionamento: 

### Participação percentual dos pedidos por estado

Será calculado o percentual de participação de cada unidade federativa em relação ao total de pedidos. Essa informação complementará a quantidade absoluta e facilitará a comparação entre os estados no dashboard.

In [ ]:
pedidos_por_estado = (
    pedidos_por_estado
    .withColumn(
        "percentual",
        F.round(
            F.col("total_pedidos") /
            F.sum("total_pedidos").over(
                Window.partitionBy()
            ) * 100,
            2
        )
    )
    .orderBy(F.desc("total_pedidos"))
)

pedidos_por_estado.show(30, truncate=False)

+--------------+-------------+----------+
|customer_state|total_pedidos|percentual|
+--------------+-------------+----------+
|SP            |41746        |41.98     |
|RJ            |12852        |12.92     |
|MG            |11635        |11.7      |
|RS            |5466         |5.5       |
|PR            |5045         |5.07      |
|SC            |3637         |3.66      |
|BA            |3380         |3.4       |
|DF            |2140         |2.15      |
|ES            |2033         |2.04      |
|GO            |2020         |2.03      |
|PE            |1652         |1.66      |
|CE            |1336         |1.34      |
|PA            |975          |0.98      |
|MT            |907          |0.91      |
|MA            |747          |0.75      |
|MS            |715          |0.72      |
|PB            |536          |0.54      |
|PI            |495          |0.5       |
|RN            |485          |0.49      |
|AL            |413          |0.42      |
|SE            |350          |0.35

### Exportação da distribuição geográfica

A tabela de pedidos por estado será exportada em CSV, contendo a sigla da unidade federativa, a quantidade de pedidos e o percentual de participação.

In [ ]:
caminho_estado = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/"
    "pedidos_por_estado"
)

(
    pedidos_por_estado
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_estado)
)

print("Exportação concluída!")
print("Destino:", caminho_estado)

Exportação concluída!
Destino: /content/drive/MyDrive/Projeto_Olist/indicadores/pedidos_por_estado


### Análise de preço e frete

Serão analisados os valores de preço dos produtos e de frete registrados na tabela de itens dos pedidos. O objetivo é conhecer a distribuição desses valores e preparar os dados para investigar a relação entre preço e frete.

In [ ]:
itens = tabelas["olist_order_items_dataset"]

itens.select(
    F.count("*").alias("total_itens"),
    F.round(F.min("price"), 2).alias("preco_minimo"),
    F.round(F.max("price"), 2).alias("preco_maximo"),
    F.round(F.avg("price"), 2).alias("preco_medio"),
    F.round(F.min("freight_value"), 2).alias("frete_minimo"),
    F.round(F.max("freight_value"), 2).alias("frete_maximo"),
    F.round(F.avg("freight_value"), 2).alias("frete_medio")
).show(truncate=False)

+-----------+------------+------------+-----------+------------+------------+-----------+
|total_itens|preco_minimo|preco_maximo|preco_medio|frete_minimo|frete_maximo|frete_medio|
+-----------+------------+------------+-----------+------------+------------+-----------+
|112650     |0.85        |6735.0      |120.65     |0.0         |409.68      |19.99      |
+-----------+------------+------------+-----------+------------+------------+-----------+



### Correlação entre preço e valor do frete

Será calculado o coeficiente de correlação de Pearson entre o preço dos produtos e o valor do frete. O resultado permitirá avaliar a intensidade e a direção da relação linear entre essas variáveis.

In [ ]:
correlacao_preco_frete = itens.stat.corr(
    "price",
    "freight_value",
    method="pearson"
)

print(
    "Correlação entre preço e frete:",
    round(correlacao_preco_frete, 4)
)

Correlação entre preço e frete: 0.4142


### Distribuição dos preços dos produtos

Serão calculados os percentis dos preços para identificar como os valores estão distribuídos. Essa informação ajudará a definir faixas de preço que representem adequadamente os itens vendidos na base.

In [ ]:
itens.select(
    F.expr(
        "percentile_approx(price, array(0.25, 0.50, 0.75, 0.90, 0.95))"
    ).alias("percentis_preco")
).show(truncate=False)

+-----------------------------------+
|percentis_preco                    |
+-----------------------------------+
|[39.9, 74.99, 134.9, 229.04, 349.9]|
+-----------------------------------+



### Agrupamento dos itens por faixa de preço

Os itens serão classificados em cinco faixas de preço, definidas com base nos percentis observados. Para cada faixa, serão calculados a quantidade de itens, o preço médio e o frete médio.

In [ ]:
itens_por_faixa_preco = (
    itens
    .withColumn(
        "faixa_preco",
        F.when(F.col("price") <= 39.90, "Até R$ 39,90")
        .when(F.col("price") <= 74.99, "R$ 39,91 a R$ 74,99")
        .when(F.col("price") <= 134.90, "R$ 75,00 a R$ 134,90")
        .when(F.col("price") <= 229.04, "R$ 134,91 a R$ 229,04")
        .when(F.col("price") <= 349.90, "R$ 229,05 a R$ 349,90")
        .otherwise("Acima de R$ 349,90")
    )
    .groupBy("faixa_preco")
    .agg(
        F.count("*").alias("total_itens"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.avg("freight_value"), 2).alias("frete_medio")
    )
    .orderBy(
        F.when(F.col("faixa_preco") == "Até R$ 39,90", 1)
        .when(F.col("faixa_preco") == "R$ 39,91 a R$ 74,99", 2)
        .when(F.col("faixa_preco") == "R$ 75,00 a R$ 134,90", 3)
        .when(F.col("faixa_preco") == "R$ 134,91 a R$ 229,04", 4)
        .when(F.col("faixa_preco") == "R$ 229,05 a R$ 349,90", 5)
        .otherwise(6)
    )
)

itens_por_faixa_preco.show(truncate=False)

+---------------------+-----------+-----------+-----------+
|faixa_preco          |total_itens|preco_medio|frete_medio|
+---------------------+-----------+-----------+-----------+
|Até R$ 39,90         |28501      |25.71      |14.36      |
|R$ 39,91 a R$ 74,99  |27940      |56.25      |16.55      |
|R$ 75,00 a R$ 134,90 |28095      |101.53     |19.99      |
|R$ 134,91 a R$ 229,04|16840      |171.27     |24.71      |
|R$ 229,05 a R$ 349,90|5769       |280.89     |28.94      |
|Acima de R$ 349,90   |5505       |713.95     |42.76      |
+---------------------+-----------+-----------+-----------+



### Exportação da análise de preço e frete

A tabela agregada por faixa de preço será exportada em CSV para utilização no dashboard. O arquivo conterá a quantidade de itens, o preço médio e o frete médio de cada faixa.

In [ ]:
caminho_preco_frete = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/preco_frete_faixas"
)

(
    itens_por_faixa_preco
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_preco_frete)
)

print("Exportação concluída:", caminho_preco_frete)

Exportação concluída: /content/drive/MyDrive/Projeto_Olist/indicadores/preco_frete_faixas


### Indicadores de desempenho logístico

Nesta etapa, serão analisadas as informações relacionadas ao processo de entrega dos pedidos. Serão utilizadas as datas de compra, entrega efetiva e previsão de entrega para avaliar o tempo de entrega e identificar possíveis atrasos. Esses indicadores ajudarão a compreender o desempenho logístico do e-commerce.

In [ ]:
pedidos = tabelas["olist_orders_dataset"]

pedidos.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



### Cálculo do tempo de entrega dos pedidos

Serão considerados apenas os pedidos com status "delivered" e com data de compra e data de entrega efetiva preenchidas. O tempo de entrega será calculado em dias, permitindo identificar a média, o menor e o maior período entre a compra e o recebimento do pedido.

In [ ]:
pedidos_entregues = (
    pedidos
    .filter(
        (F.col("order_status") == "delivered") &
        F.col("order_purchase_timestamp").isNotNull() &
        F.col("order_delivered_customer_date").isNotNull()
    )
    .withColumn(
        "tempo_entrega_dias",
        F.round(
            (
                F.col("order_delivered_customer_date").cast("long") -
                F.col("order_purchase_timestamp").cast("long")
            ) / 86400,
            2
        )
    )
)

pedidos_entregues.select(
    F.count("*").alias("total_pedidos"),
    F.round(F.avg("tempo_entrega_dias"), 2).alias("tempo_medio_entrega_dias"),
    F.round(F.min("tempo_entrega_dias"), 2).alias("menor_tempo_entrega_dias"),
    F.round(F.max("tempo_entrega_dias"), 2).alias("maior_tempo_entrega_dias")
).show(truncate=False)

+-------------+------------------------+------------------------+------------------------+
|total_pedidos|tempo_medio_entrega_dias|menor_tempo_entrega_dias|maior_tempo_entrega_dias|
+-------------+------------------------+------------------------+------------------------+
|96470        |12.56                   |0.53                    |209.63                  |
+-------------+------------------------+------------------------+------------------------+



### Distribuição dos pedidos por tempo de entrega

Os pedidos serão agrupados em faixas de duração para identificar a concentração das entregas em diferentes períodos. Essa análise permitirá visualizar quantos pedidos foram entregues em até 7 dias, entre 8 e 14 dias, entre 15 e 21 dias e acima de 21 dias.

In [ ]:
pedidos_por_tempo_entrega = (
    pedidos_entregues
    .withColumn(
        "faixa_tempo_entrega",
        F.when(F.col("tempo_entrega_dias") <= 7, "Até 7 dias")
        .when(F.col("tempo_entrega_dias") <= 14, "De 8 a 14 dias")
        .when(F.col("tempo_entrega_dias") <= 21, "De 15 a 21 dias")
        .otherwise("Acima de 21 dias")
    )
    .groupBy("faixa_tempo_entrega")
    .agg(
        F.count("*").alias("total_pedidos"),
        F.round(
            F.count("*") / F.sum(F.count("*")).over(Window.partitionBy()) * 100,
            2
        ).alias("percentual")
    )
    .orderBy(
        F.when(F.col("faixa_tempo_entrega") == "Até 7 dias", 1)
        .when(F.col("faixa_tempo_entrega") == "De 8 a 14 dias", 2)
        .when(F.col("faixa_tempo_entrega") == "De 15 a 21 dias", 3)
        .otherwise(4)
    )
)

pedidos_por_tempo_entrega.show(truncate=False)

+-------------------+-------------+----------+
|faixa_tempo_entrega|total_pedidos|percentual|
+-------------------+-------------+----------+
|Até 7 dias         |26099        |27.05     |
|De 8 a 14 dias     |40189        |41.66     |
|De 15 a 21 dias    |17697        |18.34     |
|Acima de 21 dias   |12485        |12.94     |
+-------------------+-------------+----------+



### Exportação da distribuição do tempo de entrega

A tabela com a quantidade e o percentual de pedidos por faixa de tempo de entrega será exportada em CSV para utilização no dashboard.

In [ ]:
caminho_tempo_entrega = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/pedidos_por_tempo_entrega"
)

(
    pedidos_por_tempo_entrega
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_tempo_entrega)
)

print("Exportação concluída:", caminho_tempo_entrega)

Exportação concluída: /content/drive/MyDrive/Projeto_Olist/indicadores/pedidos_por_tempo_entrega


### Classificação das entregas conforme o prazo estimado

Os pedidos entregues serão classificados como "No prazo" ou "Atrasado", considerando a comparação entre a data de entrega efetiva e a data estimada. Também será calculada a quantidade e o percentual de pedidos em cada classificação.

Pedidos sem alguma das datas necessárias para a comparação não serão classificados, evitando conclusões incorretas.

In [ ]:
entregas_classificadas = (
    pedidos
    .filter(
        (F.col("order_status") == "delivered") &
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_estimated_delivery_date").isNotNull()
    )
    .withColumn(
        "situacao_entrega",
        F.when(
            F.to_date("order_delivered_customer_date") <=
            F.to_date("order_estimated_delivery_date"),
            "No prazo"
        ).otherwise("Atrasado")
    )
)

distribuicao_prazo = (
    entregas_classificadas
    .groupBy("situacao_entrega")
    .agg(F.count("*").alias("total_pedidos"))
    .withColumn(
        "percentual",
        F.round(
            F.col("total_pedidos") /
            F.sum("total_pedidos").over(Window.partitionBy()) * 100,
            2
        )
    )
    .orderBy(F.desc("total_pedidos"))
)

distribuicao_prazo.show(truncate=False)

+----------------+-------------+----------+
|situacao_entrega|total_pedidos|percentual|
+----------------+-------------+----------+
|No prazo        |89936        |93.23     |
|Atrasado        |6534         |6.77      |
+----------------+-------------+----------+



### Exportação dos indicadores de pontualidade das entregas

A tabela com a quantidade e o percentual de pedidos entregues no prazo e com atraso será exportada em CSV para utilização no dashboard.

In [ ]:
caminho_pontualidade = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/pontualidade_entregas"
)

(
    distribuicao_prazo
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_pontualidade)
)

print("Exportação concluída:", caminho_pontualidade)

Exportação concluída: /content/drive/MyDrive/Projeto_Olist/indicadores/pontualidade_entregas


### Tempo médio de entrega por estado

Os pedidos entregues serão relacionados aos estados dos clientes. Em seguida, será calculado o tempo médio entre a compra e a entrega efetiva para cada estado. Serão considerados apenas pedidos com datas válidas, permitindo comparar o desempenho logístico entre as unidades federativas.

In [ ]:
clientes = tabelas["olist_customers_dataset"]

pedidos_com_estado = (
    pedidos_entregues
    .join(
        clientes.select("customer_id", "customer_state"),
        on="customer_id",
        how="left"
    )
)

tempo_entrega_por_estado = (
    pedidos_com_estado
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("total_pedidos"),
        F.round(F.avg("tempo_entrega_dias"), 2).alias("tempo_medio_entrega_dias")
    )
    .orderBy(F.desc("tempo_medio_entrega_dias"))
)

tempo_entrega_por_estado.show(30, truncate=False)

+--------------+-------------+------------------------+
|customer_state|total_pedidos|tempo_medio_entrega_dias|
+--------------+-------------+------------------------+
|RR            |41           |29.39                   |
|AP            |67           |27.19                   |
|AM            |145          |26.43                   |
|AL            |397          |24.54                   |
|PA            |946          |23.77                   |
|MA            |717          |21.57                   |
|SE            |335          |21.52                   |
|CE            |1279         |21.27                   |
|AC            |80           |21.04                   |
|PB            |517          |20.43                   |
|PI            |476          |19.46                   |
|RO            |243          |19.37                   |
|BA            |3256         |19.34                   |
|RN            |474          |19.28                   |
|PE            |1593         |18.45             

### Exportação do tempo médio de entrega por estado

A tabela com a quantidade de pedidos e o tempo médio de entrega por estado será exportada em CSV para utilização no dashboard.

In [ ]:
caminho_tempo_estado = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores/tempo_entrega_por_estado"
)

(
    tempo_entrega_por_estado
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(caminho_tempo_estado)
)

print("Exportação concluída:", caminho_tempo_estado)

Exportação concluída: /content/drive/MyDrive/Projeto_Olist/indicadores/tempo_entrega_por_estado


### Verificação de pedidos sem itens associados

Será realizada uma comparação entre a tabela de pedidos e a tabela de itens para identificar quantos pedidos não possuem produtos associados. Também será analisada a distribuição desses pedidos por status, permitindo compreender se a ausência de itens está relacionada a situações específicas do processo de compra.

In [ ]:
itens = tabelas["olist_order_items_dataset"]

pedidos_sem_itens = (
    pedidos
    .join(
        itens.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
)

print("Total de pedidos sem itens:", pedidos_sem_itens.count())

pedidos_sem_itens.groupBy("order_status").agg(
    F.countDistinct("order_id").alias("total_pedidos")
).orderBy(
    F.desc("total_pedidos")
).show(truncate=False)

Total de pedidos sem itens: 775
+------------+-------------+
|order_status|total_pedidos|
+------------+-------------+
|unavailable |603          |
|canceled    |164          |
|created     |5            |
|invoiced    |2            |
|shipped     |1            |
+------------+-------------+



### Observação sobre pedidos sem itens associados

Foram identificados 775 pedidos sem registros correspondentes na tabela de itens, representando aproximadamente 0,78% do total de pedidos. A maioria está classificada como indisponível (603) ou cancelada (164). Esses registros foram mantidos na base, pois representam situações existentes no processo de compra. Para as análises de receita, foram considerados os valores efetivamente registrados na tabela de itens, evitando atribuir valores financeiros sem evidência.

### Validação das bases analíticas exportadas

Será verificada a existência das pastas de exportação e dos arquivos CSV gerados pelo PySpark. Essa conferência permitirá identificar eventuais exportações ausentes ou incompletas antes da utilização dos dados no Looker Studio.

In [ ]:
import os
import glob

pasta_indicadores = (
    "/content/drive/MyDrive/Projeto_Olist/indicadores"
)

pastas_esperadas = [
    "indicadores_gerais",
    "evolucao_mensal_dashboard",
    "receita_por_categoria",
    "pedidos_por_status",
    "pedidos_por_estado",
    "preco_frete_faixas",
    "pedidos_por_tempo_entrega",
    "pontualidade_entregas",
    "tempo_entrega_por_estado"
]

print("VALIDAÇÃO DAS EXPORTAÇÕES\n")

for pasta in pastas_esperadas:
    caminho = os.path.join(pasta_indicadores, pasta)

    arquivos_csv = glob.glob(
        os.path.join(caminho, "*.csv")
    )

    if os.path.isdir(caminho) and arquivos_csv:
        situacao = "OK"
        quantidade = len(arquivos_csv)
    else:
        situacao = "VERIFICAR"
        quantidade = 0

    print(
        f"{pasta}: {situacao} | "
        f"Arquivos CSV encontrados: {quantidade}"
    )

VALIDAÇÃO DAS EXPORTAÇÕES

indicadores_gerais: OK | Arquivos CSV encontrados: 1
evolucao_mensal_dashboard: OK | Arquivos CSV encontrados: 1
receita_por_categoria: OK | Arquivos CSV encontrados: 1
pedidos_por_status: OK | Arquivos CSV encontrados: 1
pedidos_por_estado: OK | Arquivos CSV encontrados: 1
preco_frete_faixas: OK | Arquivos CSV encontrados: 1
pedidos_por_tempo_entrega: OK | Arquivos CSV encontrados: 1
pontualidade_entregas: OK | Arquivos CSV encontrados: 1
tempo_entrega_por_estado: OK | Arquivos CSV encontrados: 1


### Conferência da quantidade de registros exportados

Será realizada a leitura dos arquivos CSV exportados para conferir a quantidade de registros de cada base analítica. A validação permitirá verificar se os dados foram gravados integralmente e se os totais correspondem às tabelas utilizadas na análise.

In [ ]:


quantidades_esperadas = {
    "indicadores_gerais": 1,
    "evolucao_mensal_dashboard": 23,
    "receita_por_categoria": 0,  # Será conferida separadamente
    "pedidos_por_status": 8,
    "pedidos_por_estado": 27,
    "preco_frete_faixas": 6,
    "pedidos_por_tempo_entrega": 4,
    "pontualidade_entregas": 2,
    "tempo_entrega_por_estado": 27
}

for nome, esperado in quantidades_esperadas.items():
    caminho = os.path.join(pasta_indicadores, nome)
    arquivo = glob.glob(os.path.join(caminho, "*.csv"))[0]

    df_csv = (
        spark.read
        .option("header", True)
        .option("encoding", "UTF-8")
        .csv(arquivo)
    )

    quantidade = df_csv.count()

    if esperado == 0:
        situacao = "CONFERIR"
    else:
        situacao = "OK" if quantidade == esperado else "DIVERGENTE"

    print(
        f"{nome}: {quantidade} registros | "
        f"Esperado: {esperado if esperado else 'a verificar'} | "
        f"{situacao}"
    )

indicadores_gerais: 1 registros | Esperado: 1 | OK
evolucao_mensal_dashboard: 23 registros | Esperado: 23 | OK
receita_por_categoria: 74 registros | Esperado: a verificar | CONFERIR
pedidos_por_status: 8 registros | Esperado: 8 | OK
pedidos_por_estado: 27 registros | Esperado: 27 | OK
preco_frete_faixas: 6 registros | Esperado: 6 | OK
pedidos_por_tempo_entrega: 4 registros | Esperado: 4 | OK
pontualidade_entregas: 2 registros | Esperado: 2 | OK
tempo_entrega_por_estado: 27 registros | Esperado: 27 | OK


### Validação da tabela de receita por categoria

Será conferida a quantidade total de categorias presentes na tabela exportada, comparando o resultado com a tabela original do PySpark. Também será verificada a soma da receita para garantir que os valores financeiros foram preservados durante a exportação.

In [ ]:
arquivo_categoria = glob.glob(
    os.path.join(
        pasta_indicadores,
        "receita_por_categoria",
        "*.csv"
    )
)[0]

categorias_exportadas = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(arquivo_categoria)
)

print("Categorias exportadas:", categorias_exportadas.count())

print(
    "Categorias na tabela original:",
    receita_por_categoria.count()
)

print(
    "Receita total exportada:",
    categorias_exportadas.agg(
        F.round(F.sum("receita_total"), 2)
    ).first()[0]
)

print(
    "Receita total original:",
    receita_por_categoria.agg(
        F.round(F.sum("receita_total"), 2)
    ).first()[0]
)

Categorias exportadas: 74
Categorias na tabela original: 74
Receita total exportada: 15843553.24
Receita total original: 15843553.24


### Finalização da sessão Spark

Após concluir as análises, exportações e validações das bases analíticas, a sessão Spark será encerrada para liberar os recursos utilizados durante o processamento.

In [ ]:
spark.stop()

print("Sessão Spark encerrada com sucesso!")

Sessão Spark encerrada com sucesso!


## Conclusão da preparação das bases analíticas

A etapa de preparação das bases analíticas foi concluída com sucesso. Foram desenvolvidas e exportadas nove tabelas em formato CSV, contemplando indicadores gerais, evolução mensal, receita por categoria, pedidos por status e estado, relação entre preço e frete, tempo de entrega e pontualidade logística.

As exportações foram verificadas quanto à existência dos arquivos e à quantidade de registros. Também foi realizada a conferência da receita total por categoria, que apresentou correspondência integral com a tabela original, totalizando R$ 15.843.553,24.

Com isso, as nove bases analíticas estão disponíveis para utilização na construção do dashboard interativo no Looker Studio.